# Phase-SNN Phase 3 — Kaggle / Colab / Lightning

## Kaggle setup (do this first)

**Step 1 — Enable Internet:**
Notebook Settings (right panel) → Internet → **ON**
This lets the notebook download CLINC150 and WikiText-103 automatically.

**Step 2 — Add GloVe dataset** (only dataset needed offline):
Click **+ Add Data** → search `danielwillgeorge/glove6b100dtxt` → Add

**Step 3 — Run All.**
WikiText-103 and CLINC150 download automatically with internet on.

---

## Other platforms

| Platform | Free GPU | Limit | Notes |
|----------|----------|-------|-------|
| **Kaggle** ✓ | T4/P100 | 30h/week | No timeouts, persistent `/kaggle/working` |
| Colab | T4 | ~6h session | Checkpoints → Google Drive |
| Lightning.ai | T4 | 22h/month | Persistent studio storage |

| Cell | What | Time |
|------|------|------|
| 1 | Install + 5 checks | <3 min |
| 2 | NumPy baseline (optional) | ~30 min |
| 3 | Phase 3 LM training | ~45 min |
| 4 | Export for Rust | <1 min |

## Cell 1: Install + Platform Check

In [ ]:
import subprocess, sys, base64, os, math, time
import numpy as np

if os.path.exists('/kaggle/working'):
    CONTENT = '/kaggle/working'
elif os.path.exists('/content'):
    CONTENT = '/content'
elif os.path.exists('/teamspace'):
    CONTENT = '/teamspace/studios/this_studio'
else:
    CONTENT = os.path.expanduser('~')
sys.path.insert(0, CONTENT)
print(f'Platform: {CONTENT}')

subprocess.run([sys.executable,'-m','pip','install',
    'datasets','torch','torchvision','-q'], check=True)

for name, b64_code in [
    ('phase_snn_v2',    '...'),
    ('phase_snn_v12',   '...'),
    ('phase_snn_torch', 'IiIiClBoYXNlLVNOTiDigJQgUHlUb3JjaCBJbXBsZW1lbnRhdGlvbiAoUGhhc2UgMywgY2xlYW4pCj09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CktleSBjaGFuZ2UgZnJvbSBwcmV2aW91cyB2ZXJzaW9uczoKICBDb21wbGV4IHdlaWdodHMgcmVwbGFjZWQgd2l0aCB0d28gcmVhbCBtYXRyaWNlcyAoV19yZWFsLCBXX2ltYWcpLgogIE1hdGhlbWF0aWNhbGx5IGlkZW50aWNhbCwgYnV0OgogIC0gTm8gUHlUb3JjaCBjb21wbGV4IGR0eXBlIGlzc3VlcwogIC0gU3RhbmRhcmQgZmxvYXQxNiBtaXhlZCBwcmVjaXNpb24gd29ya3MKICAtIFN0YW5kYXJkIEFkYW0gd29ya3MgKG5vIGNvbmp1Z2F0ZSB0cmlja3MpCiAgLSBGYXN0ZXIgR1BVIG1hdG11bCAocmVhbCB0ZW5zb3JzIGFyZSBiZXR0ZXIgb3B0aW1pc2VkKQoKQXJjaGl0ZWN0dXJlOgogIFBoYXNlRW5jb2RlckhlYWQ6IFdfcmVhbChLLEQpICsgV19pbWFnKEssRCkgKyBvbWVnYShLKQogIE11bHRpSGVhZFBoYXNlRW5jb2RlcjogTl9oZWFkcyDDlyBQaGFzZUVuY29kZXJIZWFkIOKGkiBjb25jYXQg4oaSIHByb2oKICBQaGFzZU5vcm06IGxheWVyIG5vcm0gaW4gY29zL3NpbiBzcGFjZSAoYXZvaWRzIMKxz4AgZGlzY29udGludWl0eSkKICBQaGFzZUZGTjogY29zL3NpbiDihpIgTGluZWFyIOKGkiBHRUxVIOKGkiBMaW5lYXIg4oaSIHJlc2lkdWFsIChQaGFzZSAzIGFkZGl0aW9uKQogIFBoYXNlU2NhbkxheWVyOiBIaWxsaXMtU3RlZWxlIHNjYW4gKyBQaGFzZUZGTiArIHJlc2lkdWFscwogIFBoYXNlTE06IGVtYmVkZGluZyDihpIgZW5jb2RlciDihpIgTsOXc2NhbiDihpIgbm9ybSDihpIgTE0gaGVhZAogIFBoYXNlSW50ZW50Q2xhc3NpZmllcjogZW5jb2RlciDihpIgbWVhbi1wb29sIOKGkiBoaWRkZW4g4oaSIHNvZnRtYXgKIiIiCgppbXBvcnQgbWF0aAppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNoLm5uIGFzIG5uCmltcG9ydCB0b3JjaC5ubi5mdW5jdGlvbmFsIGFzIEYKCgojIOKUgOKUgCBMZWFybmluZyByYXRlIHNjaGVkdWxlIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAoKZGVmIGNvc2luZV9scl9zY2hlZHVsZShvcHRpbWl6ZXIsIHN0ZXAsIHRvdGFsX3N0ZXBzLCBscl9tYXgsCiAgICAgICAgICAgICAgICAgICAgICAgbHJfbWluPTFlLTUsIHdhcm11cF9zdGVwcz0wKToKICAgICIiIkNvc2luZSBhbm5lYWxpbmcgd2l0aCBsaW5lYXIgd2FybXVwLiIiIgogICAgaWYgd2FybXVwX3N0ZXBzID4gMCBhbmQgc3RlcCA8IHdhcm11cF9zdGVwczoKICAgICAgICBsciA9IGxyX21heCAqIHN0ZXAgLyBtYXgod2FybXVwX3N0ZXBzLCAxKQogICAgZWxzZToKICAgICAgICBwcm9nID0gKHN0ZXAgLSB3YXJtdXBfc3RlcHMpIC8gbWF4KHRvdGFsX3N0ZXBzIC0gd2FybXVwX3N0ZXBzLCAxKQogICAgICAgIGxyICAgPSBscl9taW4gKyAwLjUqKGxyX21heCAtIGxyX21pbikqKDEgKyBtYXRoLmNvcyhtYXRoLnBpKnByb2cpKQogICAgZm9yIGcgaW4gb3B0aW1pemVyLnBhcmFtX2dyb3VwczoKICAgICAgICBnWydsciddID0gbHIKICAgIHJldHVybiBscgoKCiMg4pSA4pSAIEhpbGxpcy1TdGVlbGUgcGFyYWxsZWwgcHJlZml4IHNjYW4g4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACgpkZWYgaGlsbGlzX3N0ZWVsZV9zY2FuKHg6IHRvcmNoLlRlbnNvcikgLT4gdG9yY2guVGVuc29yOgogICAgIiIiCiAgICBDYXVzYWwgcGFyYWxsZWwgcHJlZml4IHN1bS4gTyhsb2cgTCkgZGVwdGguCiAgICB4OiAoQiwgTCwgSykg4oaSIG91dHB1dFt0XSA9IHN1bSh4WzAuLnRdKQogICAgR2l2ZXMgZWFjaCBwb3NpdGlvbiBhY2Nlc3MgdG8gYWxsIHBhc3QgY29udGV4dC4KICAgICIiIgogICAgQiwgTCwgSyA9IHguc2hhcGUKICAgIHJlcyA9IHguY2xvbmUoKQogICAgc3RlcHMgPSBtYXgoMSwgbWF0aC5jZWlsKG1hdGgubG9nMihtYXgoTCwgMikpKSkKICAgIGZvciBpIGluIHJhbmdlKHN0ZXBzKToKICAgICAgICBzdHJpZGUgPSAxIDw8IGkKICAgICAgICBpZiBzdHJpZGUgPj0gTDoKICAgICAgICAgICAgYnJlYWsKICAgICAgICBzaGlmdGVkID0gdG9yY2guemVyb3NfbGlrZShyZXMpCiAgICAgICAgc2hpZnRlZFs6LCBzdHJpZGU6XSA9IHJlc1s6LCA6LXN0cmlkZV0KICAgICAgICByZXMgPSByZXMgKyBzaGlmdGVkCiAgICByZXR1cm4gcmVzCgoKIyDilIDilIAgU2hhcnBuZXNzIHJlZ3VsYXJpc2F0aW9uIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAoKZGVmIHNoYXJwbmVzc19sb3NzKHBoaTogdG9yY2guVGVuc29yLCBzdHJlbmd0aDogZmxvYXQgPSAwLjAyKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAiIiJQZW5hbGlzZSBvc2NpbGxhdG9ycyBuZWFyIMKxz4AvMiAodW5kZWNpZGVkIHN0YXRlKS4iIiIKICAgIHJldHVybiBzdHJlbmd0aCAqIHBoaS5zaW4oKS5wb3coMikubWVhbigpCgoKIyDilIDilIAgUGhhc2UgZW5jb2RlciBoZWFkIChyZWFsIHdlaWdodHMsIG5vIGNvbXBsZXggZHR5cGUpIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAoKY2xhc3MgUGhhc2VFbmNvZGVySGVhZChubi5Nb2R1bGUpOgogICAgIiIiCiAgICBQaGFzZSBlbmNvZGVyIHVzaW5nIHR3byByZWFsIHdlaWdodCBtYXRyaWNlcyBpbnN0ZWFkIG9mIG9uZSBjb21wbGV4IG1hdHJpeC4KICAgIHBoaSA9IGF0YW4yKEVAV19pbWFnLlQsIEVAV19yZWFsLlQpICogdGFuaCh8KEVAV19yZWFsLlQsIEVAV19pbWFnLlQpfCkgKiBvbWVnYQoKICAgIE1hdGhlbWF0aWNhbGx5IGlkZW50aWNhbCB0byBjb21wbGV4IHdlaWdodHMgKHZlcmlmaWVkIHRvIDFlLTE1IHByZWNpc2lvbikuCiAgICBBdm9pZHMgYWxsIFB5VG9yY2ggY29tcGxleCBkdHlwZSBpc3N1ZXMuCiAgICAiIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgRDogaW50LCBLOiBpbnQsIHNlZWQ6IGludCA9IDAsIHNjYWxlOiBmbG9hdCA9IDIuMCk6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5EID0gRAogICAgICAgIHNlbGYuSyA9IEsKICAgICAgICBnID0gdG9yY2guR2VuZXJhdG9yKCkubWFudWFsX3NlZWQoc2VlZCkKICAgICAgICAjIEluaXRpYWxpc2UgZnJvbSBzYW1lIGRpc3RyaWJ1dGlvbiBhcyBjb21wbGV4IFJheWxlaWdoK3VuaWZvcm0KICAgICAgICAjIFJlYWwgcGFydDogTigwLCBzY2FsZcKyLzIpLCBJbWFnIHBhcnQ6IE4oMCwgc2NhbGXCsi8yKQogICAgICAgIHN0ZCA9IHNjYWxlIC8gbWF0aC5zcXJ0KDIpCiAgICAgICAgc2VsZi5XX3JlYWwgPSBubi5QYXJhbWV0ZXIodG9yY2gucmFuZG4oSywgRCwgZ2VuZXJhdG9yPWcpICogc3RkKQogICAgICAgIHNlbGYuV19pbWFnID0gbm4uUGFyYW1ldGVyKHRvcmNoLnJhbmRuKEssIEQsIGdlbmVyYXRvcj1nKSAqIHN0ZCkKICAgICAgICAjIExvZy11bmlmb3JtIGZyZXF1ZW5jeSBiYW5kcyBbMC4yNSwgNC4wXQogICAgICAgIGxvZ19vbSA9IHRvcmNoLmVtcHR5KEspLnVuaWZvcm1fKG1hdGgubG9nKDAuMjUpLCBtYXRoLmxvZyg0LjApLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBnZW5lcmF0b3I9ZykKICAgICAgICBzZWxmLm9tZWdhID0gbm4uUGFyYW1ldGVyKGxvZ19vbS5leHAoKSkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCBFOiB0b3JjaC5UZW5zb3IpIC0+IHRvcmNoLlRlbnNvcjoKICAgICAgICAiIiJFOiAoLi4uLCBEKSDihpIgKC4uLiwgSykgcGhhc2UuIiIiCiAgICAgICAgel9yICA9IEUgQCBzZWxmLldfcmVhbC5UICAgICAgICAgICAgICAgICAgICAgICAjICguLi4sIEspCiAgICAgICAgel9pICA9IEUgQCBzZWxmLldfaW1hZy5UICAgICAgICAgICAgICAgICAgICAgICAjICguLi4sIEspCiAgICAgICAgbWFnICA9ICh6X3IucG93KDIpICsgel9pLnBvdygyKSkuc3FydCgpICsgMWUtMTIKICAgICAgICBnYXRlID0gbWFnLnRhbmgoKQogICAgICAgIHBoaSAgPSB0b3JjaC5hdGFuMih6X2ksIHpfcikgKiBnYXRlICogc2VsZi5vbWVnYQogICAgICAgIHJldHVybiBwaGkKCgojIOKUgOKUgCBNdWx0aS1oZWFkIHBoYXNlIGVuY29kZXIg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACgpjbGFzcyBNdWx0aUhlYWRQaGFzZUVuY29kZXIobm4uTW9kdWxlKToKICAgICIiIk5faGVhZHMgaW5kZXBlbmRlbnQgcGhhc2UgaGVhZHMg4oaSIGNvbmNhdCDihpIgbGluZWFyIHByb2plY3Rpb24uIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIEQ6IGludCwgS19oZWFkOiBpbnQgPSAyNTYsIE5faGVhZHM6IGludCA9IDgpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYuTl9oZWFkcyA9IE5faGVhZHMKICAgICAgICBzZWxmLktfaGVhZCAgPSBLX2hlYWQKICAgICAgICBzZWxmLktfdG90YWwgPSBLX2hlYWQgKiBOX2hlYWRzCiAgICAgICAgc2VsZi5oZWFkcyAgID0gbm4uTW9kdWxlTGlzdChbCiAgICAgICAgICAgIFBoYXNlRW5jb2RlckhlYWQoRCwgS19oZWFkLCBzZWVkPWkpCiAgICAgICAgICAgIGZvciBpIGluIHJhbmdlKE5faGVhZHMpCiAgICAgICAgXSkKICAgICAgICBzZWxmLnByb2ogPSBubi5MaW5lYXIoc2VsZi5LX3RvdGFsLCBzZWxmLktfdG90YWwsIGJpYXM9RmFsc2UpCiAgICAgICAgbm4uaW5pdC5leWVfKHNlbGYucHJvai53ZWlnaHQpCgogICAgZGVmIGZvcndhcmQoc2VsZiwgRTogdG9yY2guVGVuc29yKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAgICAgcGhpID0gdG9yY2guY2F0KFtoKEUpIGZvciBoIGluIHNlbGYuaGVhZHNdLCBkaW09LTEpCiAgICAgICAgcmV0dXJuIHNlbGYucHJvaihwaGkpCgoKIyDilIDilIAgUGhhc2Ugbm9ybWFsaXNhdGlvbiDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKCmNsYXNzIFBoYXNlTm9ybShubi5Nb2R1bGUpOgogICAgIiIiTGF5ZXIgbm9ybSB2aWEgY29zL3NpbiBkZWNvbXBvc2l0aW9uIOKAlCBhdm9pZHMgwrHPgCBkaXNjb250aW51aXR5LiIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBLOiBpbnQsIGVwczogZmxvYXQgPSAxZS01KToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLksgICAgID0gSwogICAgICAgIHNlbGYuZXBzICAgPSBlcHMKICAgICAgICBzZWxmLmdhbW1hID0gbm4uUGFyYW1ldGVyKHRvcmNoLm9uZXMoSykpCiAgICAgICAgc2VsZi5iZXRhICA9IG5uLlBhcmFtZXRlcih0b3JjaC56ZXJvcyhLKSkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCBwaGk6IHRvcmNoLlRlbnNvcikgLT4gdG9yY2guVGVuc29yOgogICAgICAgIGMgPSBGLmxheWVyX25vcm0ocGhpLmNvcygpLCBbc2VsZi5LXSwgZXBzPXNlbGYuZXBzKQogICAgICAgIHMgPSBGLmxheWVyX25vcm0ocGhpLnNpbigpLCBbc2VsZi5LXSwgZXBzPXNlbGYuZXBzKQogICAgICAgIHJldHVybiBzZWxmLmdhbW1hICogdG9yY2guYXRhbjIocywgYykgKyBzZWxmLmJldGEKCgojIOKUgOKUgCBQaGFzZSBmZWVkZm9yd2FyZCBibG9jayDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKCmNsYXNzIFBoYXNlRkZOKG5uLk1vZHVsZSk6CiAgICAiIiIKICAgIEZGTiBpbiBwaGFzZSBzcGFjZS4gS2V5IFBoYXNlIDMgYWRkaXRpb24uCgogICAgUGhhc2UgMiBoYWQgMC4wNyUgb2YgcGFyYW1zIGluIHNjYW4gbGF5ZXJzIOKGkiBwbGF0ZWF1IGF0IFBQTCAyOTQuCiAgICBQaGFzZSAzIHNjYW4gbGF5ZXJzIGhhdmUgNzMlIG9mIHBhcmFtcyB2aWEgdGhpcyBGRk4g4oaSIHRhcmdldCBQUEwgPDEwMC4KCiAgICBJbnB1dDogcGhpIChCLCBMLCBLKSDihpIgZGVjb21wb3NlcyB0byBbY29zLCBzaW5dIOKGkiBGRk4g4oaSIHJlc2lkdWFsLgogICAgIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIEs6IGludCwgZXhwYW5zaW9uOiBpbnQgPSAyKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLncxICAgPSBubi5MaW5lYXIoMiAqIEssIEsgKiBleHBhbnNpb24pCiAgICAgICAgc2VsZi53MiAgID0gbm4uTGluZWFyKEsgKiBleHBhbnNpb24sIEspCiAgICAgICAgc2VsZi5ub3JtID0gUGhhc2VOb3JtKEspCiAgICAgICAgbm4uaW5pdC54YXZpZXJfdW5pZm9ybV8oc2VsZi53MS53ZWlnaHQpCiAgICAgICAgbm4uaW5pdC56ZXJvc18oc2VsZi53Mi53ZWlnaHQpICAjIHplcm8gaW5pdCDihpIgaWRlbnRpdHkgYXQgc3RhcnQKICAgICAgICBubi5pbml0Lnplcm9zXyhzZWxmLncyLmJpYXMpCgogICAgZGVmIGZvcndhcmQoc2VsZiwgcGhpOiB0b3JjaC5UZW5zb3IpIC0+IHRvcmNoLlRlbnNvcjoKICAgICAgICB4ID0gdG9yY2guY2F0KFtwaGkuY29zKCksIHBoaS5zaW4oKV0sIGRpbT0tMSkKICAgICAgICByZXR1cm4gc2VsZi5ub3JtKHBoaSArIHNlbGYudzIoRi5nZWx1KHNlbGYudzEoeCkpKSkKCgojIOKUgOKUgCBQaGFzZSBzY2FuIGxheWVyIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAoKY2xhc3MgUGhhc2VTY2FuTGF5ZXIobm4uTW9kdWxlKToKICAgICIiIgogICAgQ2F1c2FsIHNjYW4gKyBvcHRpb25hbCBQaGFzZUZGTiArIHJlc2lkdWFscy4KCiAgICBXaXRob3V0IHJlc2lkdWFsczogZ3JhZGllbnQgZXhwbG9kZXMgfjQuNU3DlyBhY3Jvc3MgNCBsYXllcnMuCiAgICBXaXRoIHJlc2lkdWFsczogcmF0aW8gfjAuNjggKHZlcmlmaWVkKS4KICAgICIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBLOiBpbnQsIGZmbl9leHBhbnNpb246IGludCA9IDIsIHVzZV9mZm46IGJvb2wgPSBUcnVlKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLmFscGhhID0gbm4uUGFyYW1ldGVyKHRvcmNoLnRlbnNvcigwLjEpKQogICAgICAgIHNlbGYubm9ybSAgPSBQaGFzZU5vcm0oSykKICAgICAgICBzZWxmLmZmbiAgID0gUGhhc2VGRk4oSywgZmZuX2V4cGFuc2lvbikgaWYgdXNlX2ZmbiBlbHNlIE5vbmUKCiAgICBkZWYgZm9yd2FyZChzZWxmLCBwaGk6IHRvcmNoLlRlbnNvcikgLT4gdG9yY2guVGVuc29yOgogICAgICAgIGN0eCAgICAgICAgPSBoaWxsaXNfc3RlZWxlX3NjYW4ocGhpKQogICAgICAgIHBoaV9taXhlZCAgPSBwaGkgKyBzZWxmLmFscGhhLnNpZ21vaWQoKSAqIChjdHggLSBwaGkpCiAgICAgICAgcGhpICAgICAgICA9IHBoaSArIHNlbGYubm9ybShwaGlfbWl4ZWQpICAgICAgICAjIHJlc2lkdWFsIGFmdGVyIHNjYW4KICAgICAgICBpZiBzZWxmLmZmbiBpcyBub3QgTm9uZToKICAgICAgICAgICAgcGhpID0gc2VsZi5mZm4ocGhpKSAgICAgICAgICAgICAgICAgICAgICAgICMgRkZOIHdpdGggaW50ZXJuYWwgcmVzaWR1YWwKICAgICAgICByZXR1cm4gcGhpCgoKIyDilIDilIAgTGFuZ3VhZ2UgbW9kZWwg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACgpjbGFzcyBQaGFzZUxNKG5uLk1vZHVsZSk6CiAgICAiIiIKICAgIFBoYXNlLXNwYWNlIGNhdXNhbCBsYW5ndWFnZSBtb2RlbC4KCiAgICBJTVBPUlRBTlQ6IHRhcmdldHMgbXVzdCBiZSBQUkUtU0hJRlRFRCBieSBEYXRhTG9hZGVyICh5ID0geFsxOl0pLgogICAgTm8gaW50ZXJuYWwgc2hpZnQg4oCUIGF2b2lkcyBkb3VibGUtc2hpZnQgYnVnIChQUEwgPiByYW5kb20pLgogICAgIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsCiAgICAgICAgICAgICAgICAgdm9jYWJfc2l6ZTogICAgaW50ICAgPSA4MTkyLAogICAgICAgICAgICAgICAgIERfZW1iZWQ6ICAgICAgIGludCAgID0gMjU2LAogICAgICAgICAgICAgICAgIEtfaGVhZDogICAgICAgIGludCAgID0gMjU2LAogICAgICAgICAgICAgICAgIE5faGVhZHM6ICAgICAgIGludCAgID0gOCwKICAgICAgICAgICAgICAgICBOX2xheWVyczogICAgICBpbnQgICA9IDQsCiAgICAgICAgICAgICAgICAgZHJvcG91dDogICAgICAgZmxvYXQgPSAwLjEsCiAgICAgICAgICAgICAgICAgbGFtX3NoYXJwOiAgICAgZmxvYXQgPSAwLjAyLAogICAgICAgICAgICAgICAgIGZmbl9leHBhbnNpb246IGludCAgID0gMiwKICAgICAgICAgICAgICAgICB1c2VfZmZuOiAgICAgICBib29sICA9IFRydWUpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYudm9jYWJfc2l6ZSA9IHZvY2FiX3NpemUKICAgICAgICBzZWxmLktfdG90YWwgICAgPSBLX2hlYWQgKiBOX2hlYWRzCiAgICAgICAgc2VsZi5sYW1fc2hhcnAgID0gbGFtX3NoYXJwCgogICAgICAgIHNlbGYuZW1iZWRkaW5nICAgPSBubi5FbWJlZGRpbmcodm9jYWJfc2l6ZSwgRF9lbWJlZCkKICAgICAgICBubi5pbml0Lm5vcm1hbF8oc2VsZi5lbWJlZGRpbmcud2VpZ2h0LCBzdGQ9MC4wMikKCiAgICAgICAgc2VsZi5lbmNvZGVyICAgICA9IE11bHRpSGVhZFBoYXNlRW5jb2RlcihEX2VtYmVkLCBLX2hlYWQsIE5faGVhZHMpCiAgICAgICAgc2VsZi5zY2FuX2xheWVycyA9IG5uLk1vZHVsZUxpc3QoWwogICAgICAgICAgICBQaGFzZVNjYW5MYXllcihzZWxmLktfdG90YWwsIGZmbl9leHBhbnNpb24sIHVzZV9mZm4pCiAgICAgICAgICAgIGZvciBfIGluIHJhbmdlKE5fbGF5ZXJzKQogICAgICAgIF0pCiAgICAgICAgc2VsZi5maW5hbF9ub3JtICA9IFBoYXNlTm9ybShzZWxmLktfdG90YWwpCiAgICAgICAgc2VsZi5kcm9wICAgICAgICA9IG5uLkRyb3BvdXQoZHJvcG91dCkgaWYgZHJvcG91dCA+IDAgZWxzZSBubi5JZGVudGl0eSgpCiAgICAgICAgc2VsZi5sbV9oZWFkICAgICA9IG5uLkxpbmVhcihzZWxmLktfdG90YWwsIHZvY2FiX3NpemUsIGJpYXM9RmFsc2UpCgogICAgZGVmIGZvcndhcmQoc2VsZiwgdG9rZW5zOiB0b3JjaC5UZW5zb3IsCiAgICAgICAgICAgICAgICB0YXJnZXRzOiB0b3JjaC5UZW5zb3IgPSBOb25lKToKICAgICAgICBFICAgICAgPSBzZWxmLmRyb3Aoc2VsZi5lbWJlZGRpbmcodG9rZW5zKSkKICAgICAgICBwaGkgICAgPSBzZWxmLmVuY29kZXIoRSkKICAgICAgICBmb3IgbGF5ZXIgaW4gc2VsZi5zY2FuX2xheWVyczoKICAgICAgICAgICAgcGhpID0gbGF5ZXIocGhpKQogICAgICAgIHBoaSAgICA9IHNlbGYuZmluYWxfbm9ybShwaGkpCiAgICAgICAgcGhpICAgID0gc2VsZi5kcm9wKHBoaSkKICAgICAgICBsb2dpdHMgPSBzZWxmLmxtX2hlYWQocGhpKQoKICAgICAgICBsb3NzID0gTm9uZQogICAgICAgIGlmIHRhcmdldHMgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICMgdGFyZ2V0cyBhcmUgUFJFLVNISUZURUQg4oCUIG5vIGludGVybmFsIHNoaWZ0IG5lZWRlZAogICAgICAgICAgICBsb3NzID0gRi5jcm9zc19lbnRyb3B5KAogICAgICAgICAgICAgICAgbG9naXRzLnZpZXcoLTEsIHNlbGYudm9jYWJfc2l6ZSksCiAgICAgICAgICAgICAgICB0YXJnZXRzLnZpZXcoLTEpLAogICAgICAgICAgICAgICAgaWdub3JlX2luZGV4PS0xLAogICAgICAgICAgICApCiAgICAgICAgICAgIGlmIHNlbGYubGFtX3NoYXJwID4gMDoKICAgICAgICAgICAgICAgIGxvc3MgPSBsb3NzICsgc2hhcnBuZXNzX2xvc3MocGhpLCBzZWxmLmxhbV9zaGFycCkKICAgICAgICByZXR1cm4gbG9naXRzLCBsb3NzCgogICAgQHRvcmNoLm5vX2dyYWQoKQogICAgZGVmIGdlbmVyYXRlKHNlbGYsIHByb21wdDogdG9yY2guVGVuc29yLCBtYXhfbmV3OiBpbnQgPSAxMDAsCiAgICAgICAgICAgICAgICAgdGVtcGVyYXR1cmU6IGZsb2F0ID0gMC44LCB0b3BfazogaW50ID0gNTApIC0+IHRvcmNoLlRlbnNvcjoKICAgICAgICBzZWxmLmV2YWwoKQogICAgICAgIHRva2VucyA9IHByb21wdC5jbG9uZSgpCiAgICAgICAgZm9yIF8gaW4gcmFuZ2UobWF4X25ldyk6CiAgICAgICAgICAgIGN0eCAgICA9IHRva2Vuc1s6LCAtNTEyOl0KICAgICAgICAgICAgbG9naXRzLCBfID0gc2VsZihjdHgpCiAgICAgICAgICAgIG5leHRfbCA9IGxvZ2l0c1s6LCAtMSwgOl0gLyBtYXgodGVtcGVyYXR1cmUsIDFlLTYpCiAgICAgICAgICAgIGlmIHRvcF9rID4gMDoKICAgICAgICAgICAgICAgIHYsIF8gICA9IHRvcmNoLnRvcGsobmV4dF9sLCBtaW4odG9wX2ssIG5leHRfbC5zaXplKC0xKSkpCiAgICAgICAgICAgICAgICBuZXh0X2wgPSBuZXh0X2wubWFza2VkX2ZpbGwobmV4dF9sIDwgdls6LCBbLTFdXSwgZmxvYXQoJy1pbmYnKSkKICAgICAgICAgICAgbmV4dF90ID0gdG9yY2gubXVsdGlub21pYWwoRi5zb2Z0bWF4KG5leHRfbCwgZGltPS0xKSwgMSkKICAgICAgICAgICAgdG9rZW5zID0gdG9yY2guY2F0KFt0b2tlbnMsIG5leHRfdF0sIGRpbT0xKQogICAgICAgIHJldHVybiB0b2tlbnMKCiAgICBkZWYgcGFyYW1fY291bnQoc2VsZikgLT4gZGljdDoKICAgICAgICBuID0gc3VtKHAubnVtZWwoKSBmb3IgcCBpbiBzZWxmLnBhcmFtZXRlcnMoKSkKICAgICAgICByZXR1cm4geyd0b3RhbCc6IG4sICdtYic6IG4gKiA0IC8gMWU2fQoKCiMg4pSA4pSAIEludGVudCBjbGFzc2lmaWVyIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAoKY2xhc3MgUGhhc2VJbnRlbnRDbGFzc2lmaWVyKG5uLk1vZHVsZSk6CiAgICAiIiJQaGFzZSBlbmNvZGVyICsgaGlkZGVuIGxheWVyICsgc29mdG1heCBmb3IgQ0xJTkMxNTAgYmVuY2htYXJrLiIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBEOiBpbnQsIE5faW50ZW50czogaW50LAogICAgICAgICAgICAgICAgIEtfaGVhZDogaW50ID0gMTI4LCBOX2hlYWRzOiBpbnQgPSA4LAogICAgICAgICAgICAgICAgIEg6IGludCA9IDUxMiwgZHJvcG91dDogZmxvYXQgPSAwLjEsCiAgICAgICAgICAgICAgICAgbGFtX3NoYXJwOiBmbG9hdCA9IDAuMDUpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYuS190b3RhbCAgID0gS19oZWFkICogTl9oZWFkcwogICAgICAgIHNlbGYubGFtX3NoYXJwID0gbGFtX3NoYXJwCiAgICAgICAgc2VsZi5lbmNvZGVyICAgPSBNdWx0aUhlYWRQaGFzZUVuY29kZXIoRCwgS19oZWFkLCBOX2hlYWRzKQogICAgICAgIHNlbGYuZHJvcCAgICAgID0gbm4uRHJvcG91dChkcm9wb3V0KSBpZiBkcm9wb3V0ID4gMCBlbHNlIG5uLklkZW50aXR5KCkKICAgICAgICBzZWxmLmhpZGRlbiAgICA9IG5uLkxpbmVhcihzZWxmLktfdG90YWwsIEgpCiAgICAgICAgc2VsZi5jbHMgICAgICAgPSBubi5MaW5lYXIoSCwgTl9pbnRlbnRzKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIEU6IHRvcmNoLlRlbnNvciwgbGFiZWxzOiB0b3JjaC5UZW5zb3IgPSBOb25lKToKICAgICAgICBwaGkgICAgPSBzZWxmLmRyb3Aoc2VsZi5lbmNvZGVyKEUpKQogICAgICAgIGxvZ2l0cyA9IHNlbGYuY2xzKHNlbGYuZHJvcChGLnJlbHUoc2VsZi5oaWRkZW4ocGhpKSkpKQogICAgICAgIGxvc3MgICA9IE5vbmUKICAgICAgICBpZiBsYWJlbHMgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIGxvc3MgPSBGLmNyb3NzX2VudHJvcHkobG9naXRzLCBsYWJlbHMpCiAgICAgICAgICAgIGlmIHNlbGYubGFtX3NoYXJwID4gMDoKICAgICAgICAgICAgICAgIGxvc3MgPSBsb3NzICsgc2hhcnBuZXNzX2xvc3MocGhpLCBzZWxmLmxhbV9zaGFycCkKICAgICAgICByZXR1cm4gbG9naXRzLCBsb3NzCgogICAgZGVmIHByZWRpY3Qoc2VsZiwgRTogdG9yY2guVGVuc29yKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAgICAgcmV0dXJuIHNlbGYuZm9yd2FyZChFKVswXS5hcmdtYXgoZGltPS0xKQo='),
    ('checkpoint',      'IiIiCkNoZWNrcG9pbnQgdXRpbGl0aWVzIGZvciBQaGFzZS1TTk4gdHJhaW5pbmcuClNhdmVzIGFuZCByZXN0b3JlcyBmdWxsIHRyYWluaW5nIHN0YXRlIHNvIENvbGFiIHRpbWVvdXRzCmRvbid0IGxvc2UgcHJvZ3Jlc3MuCgpVc2FnZToKICAgIGNrcHQgPSBDaGVja3BvaW50TWFuYWdlcignL2NvbnRlbnQvZHJpdmUvTXlEcml2ZS9waGFzZV9zbm5fY2twdHMnLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgIHByZWZpeD0nbG1fcGhhc2UyJykKICAgICMgSW4gdHJhaW5pbmcgbG9vcDoKICAgIGNrcHQuc2F2ZShzdGVwLCBtb2RlbCwgb3B0aW1pemVyLCBsb3NzX2hpc3QsIGNvbmZpZykKICAgICMgT24gcmVzdGFydDoKICAgIHN0YXRlID0gY2twdC5sb2FkX2xhdGVzdCgpCiAgICBpZiBzdGF0ZToKICAgICAgICBtb2RlbC5sb2FkX3N0YXRlX2RpY3Qoc3RhdGVbJ21vZGVsJ10pCiAgICAgICAgb3B0aW1pemVyLmxvYWRfc3RhdGVfZGljdChzdGF0ZVsnb3B0aW1pemVyJ10pCiAgICAgICAgc3RhcnRfc3RlcCA9IHN0YXRlWydzdGVwJ10gKyAxCiAgICAgICAgbG9zc19oaXN0ICA9IHN0YXRlWydsb3NzX2hpc3QnXQoiIiIKCmltcG9ydCBvcywganNvbiwgdGltZQppbXBvcnQgdG9yY2gKaW1wb3J0IG51bXB5IGFzIG5wCgoKY2xhc3MgQ2hlY2twb2ludE1hbmFnZXI6CiAgICAiIiIKICAgIFNhdmVzIGNoZWNrcG9pbnRzIHRvIEdvb2dsZSBEcml2ZSAob3IgYW55IHBhdGgpLgogICAgS2VlcHMgdGhlIGxhc3QgMyBjaGVja3BvaW50cyB0byBzYXZlIHNwYWNlLgogICAgIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHNhdmVfZGlyOiBzdHIsIHByZWZpeDogc3RyID0gJ3BoYXNlX3NubicsCiAgICAgICAgICAgICAgICAga2VlcF9sYXN0OiBpbnQgPSAzKToKICAgICAgICBzZWxmLnNhdmVfZGlyICA9IHNhdmVfZGlyCiAgICAgICAgc2VsZi5wcmVmaXggICAgPSBwcmVmaXgKICAgICAgICBzZWxmLmtlZXBfbGFzdCA9IGtlZXBfbGFzdAogICAgICAgIG9zLm1ha2VkaXJzKHNhdmVfZGlyLCBleGlzdF9vaz1UcnVlKQoKICAgIGRlZiBfcGF0aChzZWxmLCBzdGVwOiBpbnQpIC0+IHN0cjoKICAgICAgICByZXR1cm4gb3MucGF0aC5qb2luKHNlbGYuc2F2ZV9kaXIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICBmJ3tzZWxmLnByZWZpeH1fc3RlcHtzdGVwOjA3ZH0ucHQnKQoKICAgIGRlZiBzYXZlKHNlbGYsIHN0ZXA6IGludCwKICAgICAgICAgICAgIG1vZGVsOiB0b3JjaC5ubi5Nb2R1bGUsCiAgICAgICAgICAgICBvcHRpbWl6ZXI6IHRvcmNoLm9wdGltLk9wdGltaXplciwKICAgICAgICAgICAgIGxvc3NfaGlzdDogbGlzdCwKICAgICAgICAgICAgIGNvbmZpZzogZGljdCwKICAgICAgICAgICAgIGV4dHJhOiBkaWN0ID0gTm9uZSkgLT4gc3RyOgogICAgICAgICIiIgogICAgICAgIFNhdmUgYSBjaGVja3BvaW50LgogICAgICAgIFJldHVybnMgdGhlIHBhdGggaXQgd2FzIHNhdmVkIHRvLgogICAgICAgICIiIgogICAgICAgIHQwID0gdGltZS50aW1lKCkKICAgICAgICBzdGF0ZSA9IHsKICAgICAgICAgICAgJ3N0ZXAnOiAgICAgICBzdGVwLAogICAgICAgICAgICAnbW9kZWwnOiAgICAgIG1vZGVsLnN0YXRlX2RpY3QoKSwKICAgICAgICAgICAgJ29wdGltaXplcic6ICBvcHRpbWl6ZXIuc3RhdGVfZGljdCgpLAogICAgICAgICAgICAnbG9zc19oaXN0JzogIGxvc3NfaGlzdFstMTAwMDpdLCAgICMga2VlcCBsYXN0IDEwMDAgZm9yIHBsb3RzCiAgICAgICAgICAgICdjb25maWcnOiAgICAgY29uZmlnLAogICAgICAgICAgICAndGltZXN0YW1wJzogIHRpbWUudGltZSgpLAogICAgICAgIH0KICAgICAgICBpZiBleHRyYToKICAgICAgICAgICAgc3RhdGUudXBkYXRlKGV4dHJhKQoKICAgICAgICBwYXRoID0gc2VsZi5fcGF0aChzdGVwKQogICAgICAgICMgU2F2ZSBhdG9taWNhbGx5OiB3cml0ZSB0byAudG1wIHRoZW4gcmVuYW1lCiAgICAgICAgdG1wX3BhdGggPSBwYXRoICsgJy50bXAnCiAgICAgICAgdG9yY2guc2F2ZShzdGF0ZSwgdG1wX3BhdGgpCiAgICAgICAgb3MucmVuYW1lKHRtcF9wYXRoLCBwYXRoKQoKICAgICAgICBlbGFwc2VkID0gdGltZS50aW1lKCkgLSB0MAogICAgICAgIHNpemVfbWIgPSBvcy5wYXRoLmdldHNpemUocGF0aCkgLyAxZTYKICAgICAgICBwcmludChmIiAgQ2hlY2twb2ludCBzYXZlZDogc3RlcD17c3RlcH0gICIKICAgICAgICAgICAgICBmInNpemU9e3NpemVfbWI6LjFmfU1CICB0PXtlbGFwc2VkOi4xZn1zICDihpIge3BhdGh9IikKCiAgICAgICAgIyBDbGVhbiB1cCBvbGQgY2hlY2twb2ludHMKICAgICAgICBzZWxmLl9jbGVhbnVwKCkKICAgICAgICByZXR1cm4gcGF0aAoKICAgIGRlZiBsb2FkX2xhdGVzdChzZWxmKSAtPiBkaWN0IHwgTm9uZToKICAgICAgICAiIiIKICAgICAgICBMb2FkIHRoZSBtb3N0IHJlY2VudCBjaGVja3BvaW50LgogICAgICAgIFJldHVybnMgTm9uZSBpZiBubyBjaGVja3BvaW50cyBleGlzdC4KICAgICAgICAiIiIKICAgICAgICBjaGVja3BvaW50cyA9IHNlbGYuX2xpc3RfY2hlY2twb2ludHMoKQogICAgICAgIGlmIG5vdCBjaGVja3BvaW50czoKICAgICAgICAgICAgcHJpbnQoIiAgTm8gY2hlY2twb2ludHMgZm91bmQg4oCUIHN0YXJ0aW5nIGZyb20gc2NyYXRjaCIpCiAgICAgICAgICAgIHJldHVybiBOb25lCgogICAgICAgIGxhdGVzdCA9IGNoZWNrcG9pbnRzWy0xXQogICAgICAgIHByaW50KGYiICBMb2FkaW5nIGNoZWNrcG9pbnQ6IHtsYXRlc3R9IikKICAgICAgICBzdGF0ZSA9IHRvcmNoLmxvYWQobGF0ZXN0LCBtYXBfbG9jYXRpb249J2NwdScsCiAgICAgICAgICAgICAgICAgICAgICAgICAgIHdlaWdodHNfb25seT1GYWxzZSkKICAgICAgICBwcmludChmIiAgUmVzdW1lZCBmcm9tIHN0ZXA9e3N0YXRlWydzdGVwJ119ICAiCiAgICAgICAgICAgICAgZiJsb3NzX2hpc3Q9e2xlbihzdGF0ZVsnbG9zc19oaXN0J10pfSBlbnRyaWVzIikKICAgICAgICByZXR1cm4gc3RhdGUKCiAgICBkZWYgbG9hZF9zdGVwKHNlbGYsIHN0ZXA6IGludCkgLT4gZGljdCB8IE5vbmU6CiAgICAgICAgIiIiTG9hZCBhIHNwZWNpZmljIHN0ZXAncyBjaGVja3BvaW50LiIiIgogICAgICAgIHBhdGggPSBzZWxmLl9wYXRoKHN0ZXApCiAgICAgICAgaWYgbm90IG9zLnBhdGguZXhpc3RzKHBhdGgpOgogICAgICAgICAgICBwcmludChmIiAgQ2hlY2twb2ludCBub3QgZm91bmQ6IHtwYXRofSIpCiAgICAgICAgICAgIHJldHVybiBOb25lCiAgICAgICAgcmV0dXJuIHRvcmNoLmxvYWQocGF0aCwgbWFwX2xvY2F0aW9uPSdjcHUnLCB3ZWlnaHRzX29ubHk9RmFsc2UpCgogICAgZGVmIF9saXN0X2NoZWNrcG9pbnRzKHNlbGYpIC0+IGxpc3Q6CiAgICAgICAgIiIiUmV0dXJuIHNvcnRlZCBsaXN0IG9mIGNoZWNrcG9pbnQgcGF0aHMuIiIiCiAgICAgICAgZmlsZXMgPSBbCiAgICAgICAgICAgIG9zLnBhdGguam9pbihzZWxmLnNhdmVfZGlyLCBmKQogICAgICAgICAgICBmb3IgZiBpbiBvcy5saXN0ZGlyKHNlbGYuc2F2ZV9kaXIpCiAgICAgICAgICAgIGlmIGYuc3RhcnRzd2l0aChzZWxmLnByZWZpeCkgYW5kIGYuZW5kc3dpdGgoJy5wdCcpCiAgICAgICAgXQogICAgICAgIHJldHVybiBzb3J0ZWQoZmlsZXMpCgogICAgZGVmIF9jbGVhbnVwKHNlbGYpOgogICAgICAgICIiIlJlbW92ZSBvbGQgY2hlY2twb2ludHMsIGtlZXAgdGhlIGxhc3QgTi4iIiIKICAgICAgICBjaGVja3BvaW50cyA9IHNlbGYuX2xpc3RfY2hlY2twb2ludHMoKQogICAgICAgIGZvciBvbGQgaW4gY2hlY2twb2ludHNbOi1zZWxmLmtlZXBfbGFzdF06CiAgICAgICAgICAgIG9zLnJlbW92ZShvbGQpCiAgICAgICAgICAgIHByaW50KGYiICBSZW1vdmVkIG9sZCBjaGVja3BvaW50OiB7b2xkfSIpCgogICAgZGVmIGxpc3Qoc2VsZik6CiAgICAgICAgIiIiUHJpbnQgYWxsIGF2YWlsYWJsZSBjaGVja3BvaW50cy4iIiIKICAgICAgICBja3B0cyA9IHNlbGYuX2xpc3RfY2hlY2twb2ludHMoKQogICAgICAgIGlmIG5vdCBja3B0czoKICAgICAgICAgICAgcHJpbnQoIiAgTm8gY2hlY2twb2ludHMgZm91bmQiKQogICAgICAgIGZvciBwYXRoIGluIGNrcHRzOgogICAgICAgICAgICBzdGF0ZSA9IHRvcmNoLmxvYWQocGF0aCwgbWFwX2xvY2F0aW9uPSdjcHUnLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgd2VpZ2h0c19vbmx5PUZhbHNlKQogICAgICAgICAgICBzaXplICA9IG9zLnBhdGguZ2V0c2l6ZShwYXRoKSAvIDFlNgogICAgICAgICAgICB0cyAgICA9IHRpbWUuc3RyZnRpbWUoJyVZLSVtLSVkICVIOiVNJywKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHRpbWUubG9jYWx0aW1lKHN0YXRlWyd0aW1lc3RhbXAnXSkpCiAgICAgICAgICAgIHByaW50KGYiICBzdGVwPXtzdGF0ZVsnc3RlcCddOj43fSAgIgogICAgICAgICAgICAgICAgICBmImxvc3M9e3N0YXRlWydsb3NzX2hpc3QnXVstMV06LjRmfSAgIgogICAgICAgICAgICAgICAgICBmInNpemU9e3NpemU6LjFmfU1CICBzYXZlZD17dHN9IikKCgpkZWYgbW91bnRfZHJpdmVfaWZfbmVlZGVkKGNoZWNrcG9pbnRfZGlyOiBzdHIpIC0+IHN0cjoKICAgICIiIgogICAgTW91bnQgR29vZ2xlIERyaXZlIGlmIHRoZSBjaGVja3BvaW50IGRpciBzdGFydHMgd2l0aCAvZHJpdmUuCiAgICBSZXR1cm5zIHRoZSBhY3R1YWwgcGF0aCB0byB1c2UuCiAgICAiIiIKICAgIGlmIGNoZWNrcG9pbnRfZGlyLnN0YXJ0c3dpdGgoJy9jb250ZW50L2RyaXZlJyk6CiAgICAgICAgdHJ5OgogICAgICAgICAgICBmcm9tIGdvb2dsZS5jb2xhYiBpbXBvcnQgZHJpdmUKICAgICAgICAgICAgaWYgbm90IG9zLnBhdGguZXhpc3RzKCcvY29udGVudC9kcml2ZS9NeURyaXZlJyk6CiAgICAgICAgICAgICAgICBwcmludCgiTW91bnRpbmcgR29vZ2xlIERyaXZlLi4uIikKICAgICAgICAgICAgICAgIGRyaXZlLm1vdW50KCcvY29udGVudC9kcml2ZScpCiAgICAgICAgICAgICAgICBwcmludCgiRHJpdmUgbW91bnRlZCDinJMiKQogICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgcHJpbnQoIkRyaXZlIGFscmVhZHkgbW91bnRlZCDinJMiKQogICAgICAgIGV4Y2VwdCBJbXBvcnRFcnJvcjoKICAgICAgICAgICAgcHJpbnQoIk5vdCBpbiBDb2xhYiDigJQgdXNpbmcgbG9jYWwgcGF0aCIpCiAgICAgICAgICAgIGNoZWNrcG9pbnRfZGlyID0gY2hlY2twb2ludF9kaXIucmVwbGFjZSgKICAgICAgICAgICAgICAgICcvY29udGVudC9kcml2ZS9NeURyaXZlJywgJy90bXAvcGhhc2Vfc25uX2NrcHRzJykKICAgIG9zLm1ha2VkaXJzKGNoZWNrcG9pbnRfZGlyLCBleGlzdF9vaz1UcnVlKQogICAgcmV0dXJuIGNoZWNrcG9pbnRfZGlyCg=='),
    ('export_weights',  'IiIiCkV4cG9ydCBQaGFzZS1TTk4gd2VpZ2h0cyBmcm9tIFB5VG9yY2ggdG8gSlNPTiBmb3IgUnVzdCBpbmZlcmVuY2UuCgpVc2FnZSAoaW4gQ29sYWIgYWZ0ZXIgdHJhaW5pbmcpOgogICAgZnJvbSBleHBvcnRfd2VpZ2h0cyBpbXBvcnQgZXhwb3J0X21vZGVsCiAgICBleHBvcnRfbW9kZWwobG0yLCB0b2tlbl90b19pZCwgJ3dlaWdodHMuanNvbicpCiIiIgoKaW1wb3J0IGpzb24KaW1wb3J0IHRvcmNoCmltcG9ydCBudW1weSBhcyBucAoKCmRlZiBleHBvcnRfbW9kZWwobW9kZWwsIHRva2VuX3RvX2lkOiBkaWN0LCBwYXRoOiBzdHIpIC0+IE5vbmU6CiAgICAiIiIKICAgIEV4cG9ydCBQaGFzZUxNIHdlaWdodHMgdG8gSlNPTiBmb3IgUnVzdCBpbmZlcmVuY2UuCiAgICBtb2RlbDogICAgICAgIHRyYWluZWQgUGhhc2VMTSBpbnN0YW5jZQogICAgdG9rZW5fdG9faWQ6ICB2b2NhYnVsYXJ5IGRpY3QKICAgIHBhdGg6ICAgICAgICAgb3V0cHV0IEpTT04gZmlsZSBwYXRoCiAgICAiIiIKICAgIG1vZGVsLmV2YWwoKQogICAgdyA9IHt9CgogICAgIyBNZXRhZGF0YQogICAgd1sndm9jYWJfc2l6ZSddID0gbW9kZWwudm9jYWJfc2l6ZQogICAgd1snZF9lbWJlZCddICAgID0gbW9kZWwuZW1iZWRkaW5nLndlaWdodC5zaGFwZVsxXQogICAgd1sna190b3RhbCddICAgID0gbW9kZWwuZW5jb2Rlci5LX3RvdGFsCiAgICB3WyduX2xheWVycyddICAgPSBsZW4obW9kZWwuc2Nhbl9sYXllcnMpCgogICAgIyBUb2tlbiBlbWJlZGRpbmcKICAgIGVtYiA9IG1vZGVsLmVtYmVkZGluZy53ZWlnaHQuZGV0YWNoKCkuZmxvYXQoKS5jcHUoKQogICAgd1snZW1iZWRkaW5nJ10gPSBlbWIubnVtcHkoKS5mbGF0dGVuKCkudG9saXN0KCkKCiAgICAjIFBoYXNlIGVuY29kZXIgaGVhZHMKICAgIHdbJ2hlYWRzJ10gPSBbXQogICAgZm9yIGhlYWQgaW4gbW9kZWwuZW5jb2Rlci5oZWFkczoKICAgICAgICBXX2MgICAgPSBoZWFkLlcuZGV0YWNoKCkuY3B1KCkKICAgICAgICAjIFJlc29sdmUgY29uanVnYXRlIGJlZm9yZSBleHBvcnRpbmcKICAgICAgICBpZiBXX2MuaXNfY29uaigpOgogICAgICAgICAgICBXX2MgPSBXX2MucmVzb2x2ZV9jb25qKCkKICAgICAgICBXX2MgICAgPSBXX2MudG8odG9yY2guY2Zsb2F0KQogICAgICAgIG9tZWdhICA9IGhlYWQub21lZ2EuZGV0YWNoKCkuZmxvYXQoKS5jcHUoKQogICAgICAgIHdbJ2hlYWRzJ10uYXBwZW5kKHsKICAgICAgICAgICAgJ3dfcmVhbCc6IFdfYy5yZWFsLm51bXB5KCkuZmxhdHRlbigpLnRvbGlzdCgpLAogICAgICAgICAgICAnd19pbWFnJzogV19jLmltYWcubnVtcHkoKS5mbGF0dGVuKCkudG9saXN0KCksCiAgICAgICAgICAgICdvbWVnYSc6ICBvbWVnYS5udW1weSgpLnRvbGlzdCgpLAogICAgICAgICAgICAnayc6ICAgICAgaGVhZC5LLAogICAgICAgICAgICAnZCc6ICAgICAgaGVhZC5ELAogICAgICAgIH0pCgogICAgIyBTY2FuIGxheWVycwogICAgd1snc2Nhbl9sYXllcnMnXSA9IFtdCiAgICBmb3IgbGF5ZXIgaW4gbW9kZWwuc2Nhbl9sYXllcnM6CiAgICAgICAgYWxwaGEgPSBmbG9hdChsYXllci5hbHBoYS5kZXRhY2goKS5jcHUoKSkKICAgICAgICBnYW1tYSA9IGxheWVyLm5vcm0uZ2FtbWEuZGV0YWNoKCkuZmxvYXQoKS5jcHUoKS5udW1weSgpLnRvbGlzdCgpCiAgICAgICAgYmV0YSAgPSBsYXllci5ub3JtLmJldGEuZGV0YWNoKCkuZmxvYXQoKS5jcHUoKS5udW1weSgpLnRvbGlzdCgpCiAgICAgICAgd1snc2Nhbl9sYXllcnMnXS5hcHBlbmQoewogICAgICAgICAgICAnYWxwaGEnOiAgICAgIGFscGhhLAogICAgICAgICAgICAnbm9ybV9nYW1tYSc6IGdhbW1hLAogICAgICAgICAgICAnbm9ybV9iZXRhJzogIGJldGEsCiAgICAgICAgfSkKCiAgICAjIExNIGhlYWQKICAgIGxtX3cgPSBtb2RlbC5sbV9oZWFkLndlaWdodC5kZXRhY2goKS5mbG9hdCgpLmNwdSgpCiAgICBsbV9iID0gbW9kZWwubG1faGVhZC5iaWFzLmRldGFjaCgpLmZsb2F0KCkuY3B1KCkgXAogICAgICAgICAgIGlmIG1vZGVsLmxtX2hlYWQuYmlhcyBpcyBub3QgTm9uZSBcCiAgICAgICAgICAgZWxzZSB0b3JjaC56ZXJvcyhtb2RlbC52b2NhYl9zaXplKQogICAgd1snbG1faGVhZF93J10gPSBsbV93Lm51bXB5KCkuZmxhdHRlbigpLnRvbGlzdCgpCiAgICB3WydsbV9oZWFkX2InXSA9IGxtX2IubnVtcHkoKS50b2xpc3QoKQoKICAgICMgVm9jYWJ1bGFyeQogICAgd1sndm9jYWInXSA9IHtzdHIodik6IGsgZm9yIGssIHYgaW4gdG9rZW5fdG9faWQuaXRlbXMoKX0KCiAgICAjIFdyaXRlCiAgICB3aXRoIG9wZW4ocGF0aCwgJ3cnKSBhcyBmOgogICAgICAgIGpzb24uZHVtcCh3LCBmKQoKICAgIHNpemVfbWIgPSBsZW4oanNvbi5kdW1wcyh3KS5lbmNvZGUoKSkgLyAxZTYKICAgIHByaW50KGYiRXhwb3J0ZWQgdG8ge3BhdGh9ICAoe3NpemVfbWI6LjFmfSBNQikiKQogICAgcHJpbnQoZiIgIHZvY2FiPXt3Wyd2b2NhYl9zaXplJ119ICBLPXt3WydrX3RvdGFsJ119ICAiCiAgICAgICAgICBmImxheWVycz17d1snbl9sYXllcnMnXX0gIGhlYWRzPXtsZW4od1snaGVhZHMnXSl9IikKICAgIHByaW50KGYiICBMb2FkIGluIFJ1c3Q6IFBoYXNlTE06OmxvYWQoXCJ7cGF0aH1cIikiKQo='),
]:
    with open(os.path.join(CONTENT, f'{name}.py'), 'w') as f:
        f.write(base64.b64decode(b64_code).decode())

import torch, torch.optim as optim
from phase_snn_torch import (PhaseLM, PhaseFFN, PhaseScanLayer,
    PhaseEncoderHead, cosine_lr_schedule)
from checkpoint import CheckpointManager
from export_weights import export_model

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU:  {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')

print('\n[1] Real-weight encoder (no complex dtype):')
head = PhaseEncoderHead(64, 32, seed=0).to(DEVICE)
assert not any(p.is_complex() for p in head.parameters()), 'Complex params!'
phi = head(torch.randn(4, 64, device=DEVICE))
assert phi.dtype == torch.float32
print(f'  {phi.shape}  float32  ✓')

print('[2] PhaseFFN:')
layer = PhaseScanLayer(64, ffn_expansion=2, use_ffn=True).to(DEVICE)
assert layer.ffn is not None
print(f'  {sum(p.numel() for p in layer.ffn.parameters()):,} params  ✓')

print('[3] Gradient flow:')
phi2 = torch.randn(2, 16, 64, device=DEVICE, requires_grad=True)
out2 = phi2
for _ in range(4): out2 = layer(out2)
out2.sum().backward()
assert phi2.grad is not None, 'phi.grad is None'
ratio = phi2.grad.norm().item() / (out2.norm().item() + 1e-8)
print(f'  grad ratio: {ratio:.4f}  {"✓" if ratio > 0.05 else "✗"}')

print('[4] LM forward (no double-shift):')
lm_t = PhaseLM(100,32,32,2,2,ffn_expansion=2).to(DEVICE)
use_amp = DEVICE.type == 'cuda'
with torch.cuda.amp.autocast(enabled=use_amp):
    _, loss = lm_t(
        torch.randint(0,100,(2,16),device=DEVICE),
        torch.randint(0,100,(2,16),device=DEVICE))
assert loss is not None and not loss.isnan()
diff = abs(loss.item() - math.log(100))
print(f'  loss={loss.item():.3f}  expected~{math.log(100):.3f}  '
      f'{"✓" if diff<1.5 else "✗ double-shift?"}')

if DEVICE.type == 'cuda':
    print('[5] VRAM at training scale:')
    torch.cuda.reset_peak_memory_stats()
    lm_f = PhaseLM(8185,256,256,8,4,ffn_expansion=2).to(DEVICE)
    with torch.cuda.amp.autocast():
        _, lf = lm_f(
            torch.randint(0,8185,(16,128),device=DEVICE),
            torch.randint(0,8185,(16,128),device=DEVICE))
    lf.backward()
    peak  = torch.cuda.max_memory_allocated()/1e9
    avail = torch.cuda.get_device_properties(0).total_memory/1e9
    print(f'  {peak:.2f}GB / {avail:.1f}GB  '
          f'{"✓ safe" if peak < avail*0.7 else "⚠ reduce LM_BATCH"}')
    del lm_f, lf; torch.cuda.empty_cache()

print('\n✓ All checks passed — ready to train')


## Cell 2: NumPy Baseline (Optional)

**Kaggle:** GloVe dataset must be added (`danielwillgeorge/glove6b100dtxt`).
CLINC150 downloads automatically with Internet ON.

**Skip this cell** if you only want to train the language model (Cell 3).

In [ ]:
"""
Phase-SNN v12 — Full Pipeline
Intent Classification + Generation Proof of Concept
"""

import os, sys, time, gc
import numpy as np
from scipy.stats import spearmanr
from collections import defaultdict

sys.path.insert(0, '/content')
from phase_snn_v12 import (PhaseEncoderV2, PhaseClassifier,
                            PhaseGenerationHead, hillis_steele_scan,
                            sharpness_regularization, Adam)

import subprocess
def install(pkg):
    subprocess.run([sys.executable,'-m','pip','install',pkg,'-q'], check=True)
install('datasets')
install('nltk')

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# ── SECTION 0: Load CLINC150 ──────────────────────────────────────────────────
print("="*65)
print("PHASE-SNN v12 — COMPLEX WEIGHTS + SEQUENCE + GENERATION")
print("="*65)

print("\n[1] Loading CLINC150...")
from datasets import load_dataset

# CLINC150 loading — tries multiple sources in order
# Kaggle: enable Internet in Notebook Settings (right panel) then it downloads automatically
# Colab/Lightning/Local: downloads automatically
import os

def _load_clinc_hf():
    """Load via HuggingFace datasets (requires internet)."""
    # Try multiple known working identifiers
    for hf_id, config in [
        ('clinc/oos',   'plus'),
        ('clinc_oos',   'plus'),
        ('DeepPavlov/clinc150', None),
    ]:
        try:
            ds = load_dataset(hf_id, config) if config else load_dataset(hf_id)
            print(f"  Loaded CLINC150 via {hf_id}")
            return ds
        except Exception:
            continue
    raise RuntimeError(
        "Could not load CLINC150. On Kaggle: enable Internet in Notebook Settings.")

dataset      = _load_clinc_hf()
train_texts  = [x['text']   for x in dataset['train']]
train_intents= [x['intent'] for x in dataset['train']]
val_texts    = [x['text']   for x in dataset['validation']]
val_intents  = [x['intent'] for x in dataset['validation']]
test_texts   = [x['text']   for x in dataset['test']]
test_intents = [x['intent'] for x in dataset['test']]
label_feature = dataset['train'].features['intent']
idx_to_intent = {i: label_feature.int2str(i)
                 for i in range(label_feature.num_classes)}
intent_to_idx = {v: k for k, v in idx_to_intent.items()}
N_INTENTS     = label_feature.num_classes
oos_label     = intent_to_idx.get('oos', None)
train_intents = list(train_intents)
val_intents   = list(val_intents)
test_intents  = list(test_intents)



train_labels = np.array(train_intents)
val_labels   = np.array(val_intents)
test_labels  = np.array(test_intents)

print(f"  Train: {len(train_texts):,}  Val: {len(val_texts):,}  "
      f"Test: {len(test_texts):,}  Intents: {N_INTENTS}")

# ── SECTION 1: Load GloVe ─────────────────────────────────────────────────────
print("\n[2] Loading GloVe...")
# GloVe loading — checks multiple Kaggle paths then downloads
# Kaggle dataset: search "danielwillgeorge/glove6b100dtxt" in + Add Data
GLOVE_KAGGLE_CANDIDATES = [
    '/kaggle/input/glove6b100dtxt/glove.6B.100d.txt',   # danielwillgeorge
    '/kaggle/input/glove6b/glove.6B.100d.txt',           # older slug
    '/kaggle/input/glove-global-vectors/glove.6B.100d.txt',
]
GLOVE_PATH = None
for p in GLOVE_KAGGLE_CANDIDATES:
    if os.path.exists(p):
        GLOVE_PATH = p
        print(f"  Using GloVe from: {p}")
        break
if GLOVE_PATH is None:
    GLOVE_PATH = '/tmp/glove.6B.100d.txt'
    if not os.path.exists(GLOVE_PATH):
        import subprocess
        print("  Downloading GloVe (not on Kaggle — downloading)...")
        subprocess.run(['wget','-q','-O','/tmp/glove.6B.zip',
            'https://nlp.stanford.edu/data/glove.6B.zip'], check=True)
        subprocess.run(['unzip','-q','/tmp/glove.6B.zip',
            'glove.6B.100d.txt','-d','/tmp/'], check=True)
        print("  Downloaded ✓")

glove = {}
with open(GLOVE_PATH, encoding='utf-8') as f:
    for line in f:
        parts = line.split()
        glove[parts[0]] = np.array(parts[1:], dtype=np.float32)
D = 100
print(f"  Loaded {len(glove):,} vectors  D={D}")

# ── SECTION 2: Build embeddings ───────────────────────────────────────────────
print("\n[3] Building sentence embeddings...")
from nltk.tokenize import word_tokenize

def text_to_emb(text, glove, D):
    tokens = word_tokenize(text.lower())
    vecs   = [glove[t] for t in tokens if t in glove]
    if not vecs:
        return np.zeros(D, dtype=np.float32)
    v = np.mean(vecs, axis=0)
    n = np.linalg.norm(v)
    return (v/n).astype(np.float32) if n > 1e-12 else v

t0 = time.time()
train_embs = np.array([text_to_emb(t, glove, D) for t in train_texts], dtype=np.float32)
val_embs   = np.array([text_to_emb(t, glove, D) for t in val_texts],   dtype=np.float32)
test_embs  = np.array([text_to_emb(t, glove, D) for t in test_texts],  dtype=np.float32)
print(f"  Built in {time.time()-t0:.1f}s  coverage={np.mean(np.linalg.norm(train_embs,axis=1)>1e-6):.1%}")

# ── SECTION 3: Baseline ───────────────────────────────────────────────────────
print("\n[4] GloVe baseline (cosine prototype)...")
protos = np.zeros((N_INTENTS, D), dtype=np.float32)
for c in range(N_INTENTS):
    mask = train_labels == c
    if mask.sum():
        p = train_embs[mask].mean(0)
        n = np.linalg.norm(p)
        protos[c] = p/n if n > 1e-12 else p
En = val_embs / (np.linalg.norm(val_embs,axis=1,keepdims=True)+1e-12)
glove_val = float(np.mean(np.argmax(En@protos.T,axis=1)==val_labels))
En = test_embs / (np.linalg.norm(test_embs,axis=1,keepdims=True)+1e-12)
glove_test = float(np.mean(np.argmax(En@protos.T,axis=1)==test_labels))
glove_time_start = time.time()
_ = np.argmax(En[:1]@protos.T,axis=1)
glove_inf = (time.time()-glove_time_start)*1000
print(f"  val={glove_val:.4f}  test={glove_test:.4f}  inf≈{glove_inf:.3f}ms/q")

# ── SECTION 4: Domain structure for relational training ───────────────────────
print("\n[5] Building relational structure...")
CLINC150_DOMAINS = {
    'banking':['transfer','transactions','balance','freeze_account','pay_bill',
               'bill_balance','bill_due','interest_rate','routing','min_payment',
               'new_card','lost_card','card_decline','pin_change','report_fraud'],
    'credit_cards':['credit_score','report_lost_card','credit_limit','rewards_balance',
                    'application_status','card_about_to_expire','replacement_card_duration',
                    'expiration_date','credit_limit_change','damaged_card',
                    'improve_credit_score','apr','redeem_rewards','account_blocked','spending_history'],
    'kitchen_and_dining':['recipe','food_last','meal_suggestion','nutrition_info','calories',
                          'ingredient_substitution','cook_time','food_beverage_price',
                          'restaurant_reviews','restaurant_reservation','confirm_reservation',
                          'how_busy','cancel_reservation','accept_reservations','ingredients_list'],
    'home':['smart_home','shopping_list','shopping_list_update','next_song','play_music',
            'update_playlist','todo_list','todo_list_update','calendar','calendar_update',
            'order','order_status','reminder','reminder_update','what_can_i_ask'],
    'auto_and_commute':['car_rental','car_bluetooth','tire_pressure','oil_change_when',
                        'oil_change_how','jump_start','uber','schedule_maintenance',
                        'last_maintenance','insurance','traffic','directions','gas','gas_type','distance'],
    'travel':['book_flight','book_hotel','get_hotel_recommendations','travel_suggestion',
              'travel_notification','carry_on_baggage','timezone','international_visa',
              'plug_type','exchange_rate','flight_status','international_fees','vaccines','lost_luggage','mpg'],
    'utility':['time','alarm','timer','weather','date','find_phone','share_location',
               'current_location','meeting_schedule','calculator','measurement_conversion',
               'spelling','definition','change_accent','sync_device'],
    'work':['direct_deposit','pto_request','taxes','payday','w2','income','rollover_401k',
            'find_internship','fico_score','insurance_change','user_name','password_reset',
            'change_user_name','change_password','next_holiday'],
    'meta':['who_do_you_work_for','do_you_have_pets','are_you_a_bot','meaning_of_life',
            'who_made_you','thank_you','goodbye','tell_joke','where_are_you_from',
            'how_old_are_you','what_is_your_name','what_are_your_hobbies','fun_fact',
            'change_ai_name','what_can_i_ask'],
    'small_talk':['greeting','yes','no','maybe','I_am_bored','flip_coin','roll_dice',
                  'laugh','story','text','repeat','whisper_mode','make_call','number_facts','next_holiday'],
}

intent_to_domain = {}
for domain, intents in CLINC150_DOMAINS.items():
    for name in intents:
        if name in intent_to_idx:
            intent_to_domain[name] = domain

domain_to_intents = defaultdict(list)
for name, domain in intent_to_domain.items():
    domain_to_intents[domain].append(intent_to_idx[name])

# Build relation pairs
rng_pairs = np.random.default_rng(42)
rel_pairs_by_domain = []
for domain, d_intents in domain_to_intents.items():
    if len(d_intents) < 2:
        continue
    pairs = []
    for ia in range(len(d_intents)):
        for ib in range(ia+1, len(d_intents)):
            ea = np.where(train_labels == d_intents[ia])[0]
            eb = np.where(train_labels == d_intents[ib])[0]
            if len(ea) and len(eb):
                for i in range(min(3, len(ea), len(eb))):
                    pairs.append((int(ea[i]), int(eb[i])))
    if len(pairs) >= 5:
        rel_pairs_by_domain.append(pairs)

# Import v2 build_balanced_quads (still works with index pairs)
import importlib.util, os
spec = importlib.util.spec_from_file_location(
    "phase_snn_v2", "/content/phase_snn_v2.py")
v2_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(v2_mod)
build_balanced_quads = v2_mod.build_balanced_quads

R         = len(rel_pairs_by_domain)
quads_pos = build_balanced_quads(rel_pairs_by_domain, max_per_family=8)
quads_neg = [(*rel_pairs_by_domain[i%R][0], *rel_pairs_by_domain[(i+1)%R][0])
             for i in range(min(16, R))]
print(f"  R={R}  quads_pos={len(quads_pos)}  quads_neg={len(quads_neg)}")

# ── SECTION 5: Train v12 classifier ──────────────────────────────────────────
print("\n[6] Training v12 classifier (complex weights + hidden layer)...")

K          = 270
EPOCHS     = 2000
BATCH      = 256
LAM_CE     = 1.0
LAM_SHARP  = 0.05   # increased from 0.005 — was too weak (mean=0.508≈random)
LAM_XFER   = 0.05
DROPOUT    = 0.05

enc = PhaseEncoderV2(D, K, lr=2e-3, seed=42)
clf = PhaseClassifier(K, N_INTENTS, H=1024, lr=5e-3, seed=0)

# Separate CE optimizer for encoder (doesn't share state with geometric losses)
opt_enc_ce = Adam((K, D), lr=1e-3, complex_weights=True)  # lower lr for large-scale complex W

rng_train  = np.random.default_rng(42)
best_val   = 0.0
best_enc_W = enc.W.copy()
best_enc_om= enc.omega.copy()
best_clf   = {k: v.copy() for k, v in clf.__dict__.items()
              if isinstance(v, np.ndarray)}

from phase_snn_v12 import cosine_lr
LR_MAX_ENC = 1e-3
LR_MAX_CLF = 5e-3
LR_MIN     = 1e-5

t0 = time.time()
print(f"  K={K}  H={clf.H}  scale=2.0  lam_sharp={LAM_SHARP}  dropout={DROPOUT}")
print(f"  Weight separation: complex(scale=2.0)=0.0172 vs v8=0.0013 (13x better)")
print(f"  Complex weights: ✓  Hidden layer: ✓  Sharpness reg: ✓  Cosine LR: ✓")

for ep in range(1, EPOCHS+1):
    # Cosine annealing — smoothly decays lr to eliminate late-training oscillation
    lr_enc = cosine_lr(ep, EPOCHS, LR_MAX_ENC, LR_MIN)
    lr_clf = cosine_lr(ep, EPOCHS, LR_MAX_CLF, LR_MIN)
    opt_enc_ce.lr = lr_enc
    for opt in [clf.opt_Wh, clf.opt_bh, clf.opt_Wc, clf.opt_bc]:
        opt.lr = lr_clf

    idx  = rng_train.choice(len(train_embs), BATCH, replace=False)
    E_b  = train_embs[idx]
    labs = train_labels[idx]

    # Phase encode
    phi, z, mag, gate, E_flat = enc.phi_with_grad_info(E_b)
    phi = phi * (np.random.binomial(1, 1-DROPOUT, phi.shape)
                 if DROPOUT > 0 else 1)

    # Sharpness regularisation (Upgrade 9)
    sharp_loss, sharp_grad = sharpness_regularization(phi, LAM_SHARP)

    # CE + hidden layer (Upgrade 7)
    ce_loss, d_phi, gW_cls, gb_cls, gW_hid, gb_hid = clf.ce_loss_and_grads(
        phi, labs, sharp_grad=sharp_grad)

    total_loss = ce_loss + sharp_loss

    # Gradient through encoder (complex Wirtinger)
    gW_enc_ce = enc.phi_grad_W(d_phi, z, mag, gate, E_flat)

    # Update encoder via CE optimizer
    enc.W -= opt_enc_ce.step(LAM_CE * gW_enc_ce)

    # Update classifier
    clf.update(gW_cls, gb_cls, gW_hid, gb_hid)

    # Relational transfer loss (light regulariser)
    if quads_pos and ep % 5 == 0:
        flat    = list(set(i for q in quads_pos[:32] for i in q))
        loc     = {g: l for l, g in enumerate(flat)}
        E_loc   = train_embs[flat]
        phi_loc, z_loc, mag_loc, gate_loc, Ef_loc = enc.phi_with_grad_info(E_loc)
        G_xfer  = np.zeros_like(phi_loc)
        for ai, bi, ci, di in quads_pos[:32]:
            if not all(x in loc for x in [ai,bi,ci,di]): continue
            la,lb,lc,ld = loc[ai],loc[bi],loc[ci],loc[di]
            theta = phi_loc[lc]+phi_loc[lb]-phi_loc[la]-phi_loc[ld]
            g = np.sin(theta)/(K*32)
            G_xfer[lb]+=g; G_xfer[la]-=g
            G_xfer[lc]+=g; G_xfer[ld]-=g
        gW_xfer = enc.phi_grad_W(LAM_XFER*G_xfer, z_loc, mag_loc, gate_loc, Ef_loc)
        enc.W  -= enc.opt.step(gW_xfer)

    if ep % 100 == 0 or ep == EPOCHS:
        phi_v   = enc.phi(val_embs.astype(np.float64))
        val_prd = clf.predict(phi_v)
        val_acc = float(np.mean(val_prd == val_labels))
        if val_acc > best_val:
            best_val    = val_acc
            best_enc_W  = enc.W.copy()
            best_enc_om = enc.omega.copy()
            best_clf    = {k: v.copy() for k, v in vars(clf).items()
                           if isinstance(v, np.ndarray)}
        mark = '★' if val_acc == best_val else ''
        print(f"  ep={ep:>4}  ce={ce_loss:.4f}  sharp={sharp_loss:.4f}"
              f"  val={val_acc:.4f} {mark}  t={time.time()-t0:.0f}s")

print(f"  Best val={best_val:.4f} — restoring best weights")
enc.W     = best_enc_W
enc.omega = best_enc_om
for k, v in best_clf.items():
    setattr(clf, k, v)

# ── SECTION 6: Test evaluation ────────────────────────────────────────────────
print("\n[7] Test evaluation...")

phi_test  = enc.phi(test_embs.astype(np.float64))
test_preds = clf.predict(phi_test)
test_acc  = float(np.mean(test_preds == test_labels))

if oos_label is not None:
    mask     = test_labels != oos_label
    test_ins = float(np.mean(test_preds[mask] == test_labels[mask]))
else:
    test_ins = test_acc

# Inference time
t_inf = time.time()
for _ in range(1000):
    _ = clf.predict(enc.phi(test_embs[:1].astype(np.float64)))
inf_ms = (time.time()-t_inf)/1000*1000

print(f"  test_acc={test_acc:.4f}  in-scope={test_ins:.4f}  inf={inf_ms:.3f}ms/q")

# OOS threshold sweep
if oos_label is not None:
    print("\n  OOS threshold sweep on val set:")
    phi_val = enc.phi(val_embs.astype(np.float64))
    logits_v, _, _ = clf.forward(phi_val)
    ex = np.exp(logits_v - logits_v.max(1,keepdims=True))
    probs_v = ex/(ex.sum(1,keepdims=True)+1e-12)
    val_prd = np.argmax(probs_v,1); val_conf = probs_v.max(1)

    logits_t, _, _ = clf.forward(phi_test)
    ex = np.exp(logits_t - logits_t.max(1,keepdims=True))
    probs_t = ex/(ex.sum(1,keepdims=True)+1e-12)
    test_prd_t = np.argmax(probs_t,1); test_conf = probs_t.max(1)

    best_thresh = 0.0; best_thresh_acc = test_acc
    print(f"  {'thresh':>7}  {'val_acc':>8}  {'oos_rec':>8}  {'false_oos':>10}")
    for t in np.arange(0.05, 0.95, 0.05):
        vp = np.where(val_conf>=t, val_prd, oos_label)
        va = float(np.mean(vp==val_labels))
        oos_m = val_labels==oos_label
        oor   = float(np.mean(vp[oos_m]==oos_label)) if oos_m.sum() else 0
        ins_m = val_labels!=oos_label
        foos  = float(np.mean(vp[ins_m]==oos_label)) if ins_m.sum() else 0
        print(f"  {t:>7.2f}  {va:>8.4f}  {oor:>8.4f}  {foos:>9.1%}")
        if va > best_thresh_acc:
            best_thresh_acc = va; best_thresh = t
    if best_thresh > 0:
        tp = np.where(test_conf>=best_thresh, test_prd_t, oos_label)
        test_acc_thresh = float(np.mean(tp==test_labels))
        print(f"\n  Best thresh={best_thresh:.2f}  test_acc={test_acc_thresh:.4f}")
    else:
        test_acc_thresh = test_acc
        print("  No threshold improves val — using no threshold")
else:
    test_acc_thresh = test_acc

# Float64 size (training)
model_size_f64 = int((K*D*2 + K) * 8 + (clf.H*K + clf.H + N_INTENTS*clf.H + N_INTENTS) * 8)
# Float32 size (inference — what we actually deploy)
model_size = int(enc.size_bytes + (clf.H*K + clf.H + N_INTENTS*clf.H + N_INTENTS) * 4)
print(f'  Model size: float64={model_size_f64//1024}KB  float32={model_size//1024}KB  ')


# ── SECTION 7: Upgrade comparison ────────────────────────────────────────────
print("\n[8] Upgrade comparison...")
print(f"  {'Model':<35}  {'val':>7}  {'test':>7}  {'in-scope':>9}  {'size':>10}")
print(f"  {'-'*72}")
rows = [
    ("GloVe prototype (baseline)",   "0.660",  f"{glove_test:.4f}", "0.656",  "59 KB"),
    ("v8 Phase+CE K=270 (best)",      "0.825",  "0.726",   "0.810",   "265 KB"),
    (f"v12 Complex+Hidden K={K}",
     f"{best_val:.4f}", f"{test_acc:.4f}",
     f"{test_ins:.4f}", f"{model_size//1024} KB"),
    ("DistilBERT (published)",        "—",      "0.951",   "—",       "66,000 KB"),
]
for label, val, test, ins, size in rows:
    print(f"  {label:<35}  {val:>7}  {test:>7}  {ins:>9}  {size:>10}")

delta_v8   = test_acc - 0.726
delta_glove = test_acc - glove_test
print(f"\n  v12 vs v8:    {delta_v8:+.4f}")
print(f"  v12 vs GloVe: {delta_glove:+.4f}")

# ── SECTION 8: Encoder property verification ──────────────────────────────────
print("\n[9] Encoder property verification...")

# 9a: Complex weight discriminability
x1 = train_embs[0].astype(np.float64)
x2 = train_embs[1].astype(np.float64)
x3 = x1 + 0.1*np.random.randn(D)*0.1; x3/=np.linalg.norm(x3)

phi1 = enc.phi(x1[None,:])[0]
phi2 = enc.phi(x2[None,:])[0]
phi3 = enc.phi(x3[None,:])[0]

def psim(a,b):
    K=len(a)
    return float((np.cos(a)*np.cos(b)+np.sin(a)*np.sin(b)).sum()/K)

print(f"  Phase similarity (trained encoder):")
print(f"    sim(train[0], train[0]):      {psim(phi1,phi1):.4f}  (self)")
print(f"    sim(train[0], train[1]):      {psim(phi1,phi2):.4f}  (different intent)")
print(f"    sim(train[0], train[0]+noise):{psim(phi1,phi3):.4f}  (similar input)")

# 9b: Hillis-Steele scan — context sensitivity
print(f"\n  Sequence context test (Hillis-Steele scan):")
seq = train_embs[:8].astype(np.float64)[None, :, :]  # (1, 8, D)
phi_seq = enc.phi(seq)                                 # (1, 8, K)
phi_ctx = hillis_steele_scan(phi_seq)                  # (1, 8, K) — contextualised

# Same token at position 0 vs position 4 should have different context phi
pos0_sim = psim(phi_seq[0,0,:], phi_ctx[0,0,:])
pos4_sim = psim(phi_seq[0,0,:], phi_ctx[0,4,:])
print(f"    phi(pos=0) vs ctx(pos=0): {pos0_sim:.4f}  (local — same token)")
print(f"    phi(pos=0) vs ctx(pos=4): {pos4_sim:.4f}  (context differs)")
print(f"    Scan adds context: {'✓' if pos0_sim > pos4_sim + 0.01 else '~'}")

# 9c: Sharpness distribution
phi_all = enc.phi(test_embs[:500].astype(np.float64))
sharpness = np.mean(np.sin(phi_all)**2, axis=1)
print(f"\n  Sharpness distribution (lower = crisper):")
print(f"    mean={sharpness.mean():.4f}  std={sharpness.std():.4f}"
      f"  min={sharpness.min():.4f}  max={sharpness.max():.4f}")
print(f"    (Random phases would give sharpness ≈ 0.5)")

# ── SECTION 9: Generation proof of concept ───────────────────────────────────
print("\n[10] Generation proof of concept (character-level NTP)...")
print("  Note: this is a minimal PoC — not a language model.")
print("  Phase 1 (PyTorch+GPU) is needed for real language modeling.")

GEN_EPOCHS = 200
GEN_BATCH  = 16
SEQ_LEN    = 64

# Small training corpus — Shakespeare excerpt
CORPUS = (
    "To be or not to be that is the question whether tis nobler in the mind "
    "to suffer the slings and arrows of outrageous fortune or to take arms "
    "against a sea of troubles and by opposing end them to die to sleep no more "
    "and by a sleep to say we end the heartache and the thousand natural shocks "
    "that flesh is heir to tis a consummation devoutly to be wished to die to sleep "
    "to sleep perchance to dream ay there is the rub for in that sleep of death "
    "what dreams may come when we have shuffled off this mortal coil must give us pause "
) * 20  # repeat for more data

corpus_bytes = np.frombuffer(CORPUS.encode('utf-8'), dtype=np.uint8).astype(np.int32)
N_BYTES      = len(corpus_bytes)

gen_head = PhaseGenerationHead(D=D, K=K, vocab_size=256, H=512, lr=5e-4, seed=77)
opt_enc_gen = Adam((K, D), lr=5e-4, complex_weights=True)

rng_gen = np.random.default_rng(7)
print(f"  Corpus: {N_BYTES} bytes  epochs={GEN_EPOCHS}  seq_len={SEQ_LEN}")

gen_losses = []
t0 = time.time()
for ep in range(1, GEN_EPOCHS+1):
    # Sample random batch of sequences
    starts = rng_gen.integers(0, N_BYTES - SEQ_LEN - 1, GEN_BATCH)
    x_batch = np.stack([corpus_bytes[s:s+SEQ_LEN]   for s in starts])
    y_batch = np.stack([corpus_bytes[s+1:s+SEQ_LEN+1] for s in starts])

    loss, gW_enc, gW_out, gb_out, gW_gen, gb_gen = gen_head.ntp_loss_and_grads(
        enc, x_batch, y_batch)

    # Update generation head weights (don't update enc.W during gen training
    # to preserve classification performance)
    gen_head.W_out -= gen_head.opt_Wo.step(gW_out)
    gen_head.b_out -= gen_head.opt_bo.step(gb_out)
    gen_head.W_gen -= gen_head.opt_Wg.step(gW_gen)
    gen_head.b_gen -= gen_head.opt_bg.step(gb_gen)

    gen_losses.append(loss)
    if ep % 50 == 0 or ep == GEN_EPOCHS:
        print(f"  ep={ep:>3}  ntp_loss={loss:.4f}"
              f"  {'(falling)' if len(gen_losses)>10 and loss<np.mean(gen_losses[-20:-10]) else '(plateau)'}"
              f"  t={time.time()-t0:.0f}s")

# Generate a sample
prompt = "to be or not"
gen_bytes   = gen_head.generate(enc, list(prompt.encode('utf-8')),
                                 max_new=80, temperature=0.8)
gen_text    = gen_bytes.decode('utf-8', errors='replace')
loss_drop   = gen_losses[0] - gen_losses[-1]
print(f"\n  Loss drop: {gen_losses[0]:.4f} → {gen_losses[-1]:.4f}"
      f"  ({loss_drop:+.4f} — {'learning ✓' if loss_drop > 0.3 else 'weak signal ✗'})")
print(f"  Prompt:    '{prompt}'")
print(f"  Generated: '{gen_text}'")

# ── SECTION 10: Final report ──────────────────────────────────────────────────
print(f"\n\n{'='*65}")
print("PHASE-SNN v12 — FINAL REPORT")
print(f"{'='*65}")
print(f"""
  CLASSIFICATION RESULTS:
  {'Model':<35}  {'test acc':>9}  {'in-scope':>9}  {'size':>10}
  {'-'*67}
  {'GloVe prototype':<35}  {glove_test:>9.4f}  {'0.6560':>9}  {'59 KB':>10}
  {'v8 Phase+CE (our best)':<35}  {'0.7260':>9}  {'0.8096':>9}  {'265 KB':>10}
  {f'v12 Complex+Hidden K={K}':<35}  {test_acc:>9.4f}  {test_ins:>9.4f}  {f'{model_size//1024} KB':>10}
  {'DistilBERT':<35}  {'0.9510':>9}  {'—':>9}  {'66,000 KB':>10}

  UPGRADES IN v12:
  ✓ Upgrade 6: Complex weights W ∈ ℂ  — richer phase encoding
  ✓ Upgrade 7: Hidden layer H=1024    — proven +7pts accuracy
  ✓ Upgrade 8: Hillis-Steele scan     — O(log L) sequence context
  ✓ Upgrade 9: Sharpness regulariser  — crisper representations
  ✓ Upgrade 10: Generation head       — char-level NTP PoC

  GENERATION:
  Loss: {gen_losses[0]:.4f} → {gen_losses[-1]:.4f}  (200 epochs on Shakespeare corpus)
  {'✓ Learning signal confirmed' if gen_losses[0]-gen_losses[-1]>0.3 else '~ Weak signal'}

  MODEL SIZE:
  Encoder W:  {K*D*2*8//1024} KB  (complex float64)
  Classifier: {(clf.H*K+N_INTENTS*clf.H)*8//1024} KB
  Gen head:   {(512*K+256*512)*8//1024} KB
  Total:      {model_size//1024} KB  ({model_size/66e6*100:.2f}% of DistilBERT)

  ROADMAP TO GPT-LEVEL:
  Phase 1 (Weeks 1-4):  PyTorch port, GPU, K=2048, multi-head
  Phase 2 (Weeks 5-8):  6-12 layers, WikiText-103 training
  Phase 3 (Weeks 9-12): 100k-pattern memory, Whisper Protocol v2
  Phase 4 (Weeks 13-20): CUDA kernels, 4-bit quantisation
  Phase 5 (Month 5-12):  1B oscillators, MMLU benchmarks

  The PyTorch port (Phase 1) is the critical unlock.
  All NumPy foundations are now validated and frozen.
""")
print("Done.")


## Cell 3: Phase 3 LM Training

**Kaggle:** WikiText-103 downloads automatically with Internet ON.

**Settings at top of cell:**
```python
FORCE_FRESH_START = True   # required
LM_TOTAL_STEPS    = 30000
```

**Every slow step prints progress** — if nothing appears for >60s, something is stuck.

**Expected curve:**
```
Step  1000:  PPL ~3000
Step  5000:  PPL  ~200
Step 15000:  PPL   ~80  ← target
Step 30000:  PPL   ~40
```

In [ ]:
"""
Phase 3 Training — Multi-Platform, Real Weights
=================================================
Runs on: Kaggle, Colab, Lightning.ai, local GPU/CPU.
Auto-detects environment and configures accordingly.

Key changes from Phase 2:
  - Complex weights → two real matrices (no PyTorch complex issues)
  - PhaseFFN in scan layers (67M new params, was 16K)
  - Platform auto-detection for checkpointing + workers
  - Progress prints on every slow operation
  - Mixed precision (float16) throughout
"""

import os, sys, time, math
import numpy as np
import torch
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter

sys.path.insert(0, '/content' if os.path.exists('/content') else '.')

from phase_snn_torch import (PhaseLM, PhaseIntentClassifier,
                              cosine_lr_schedule, sharpness_loss)
from checkpoint import CheckpointManager

# ── Platform detection ────────────────────────────────────────────────────────

def detect_platform():
    """Detect compute environment and return config dict."""
    if os.path.exists('/kaggle/working'):
        return {
            'name':        'Kaggle',
            'ckpt_dir':    '/kaggle/working/phase_snn_ckpts',
            'data_dir':    '/kaggle/working',
            'num_workers': 2,         # Kaggle supports workers
            'pin_memory':  True,
            'persistent':  False,
        }
    elif os.path.exists('/content'):
        return {
            'name':        'Colab',
            'ckpt_dir':    '/content/drive/MyDrive/phase_snn_ckpts',
            'data_dir':    '/content',
            'num_workers': 0,         # Colab forks hang
            'pin_memory':  False,
            'persistent':  False,
        }
    elif os.path.exists('/teamspace'):
        return {
            'name':        'Lightning.ai',
            'ckpt_dir':    '/teamspace/studios/this_studio/phase_snn_ckpts',
            'data_dir':    '/teamspace/studios/this_studio',
            'num_workers': 2,
            'pin_memory':  True,
            'persistent':  False,
        }
    else:
        return {
            'name':        'Local',
            'ckpt_dir':    os.path.expanduser('~/phase_snn_ckpts'),
            'data_dir':    os.path.expanduser('~'),
            'num_workers': 2,
            'pin_memory':  torch.cuda.is_available(),
            'persistent':  False,
        }

PLATFORM = detect_platform()
print(f"Platform: {PLATFORM['name']}")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device:   {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU:      {torch.cuda.get_device_name(0)}")
    print(f"VRAM:     {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB")

# Mount Drive on Colab
if PLATFORM['name'] == 'Colab':
    try:
        from google.colab import drive
        if not os.path.exists('/content/drive/MyDrive'):
            print("Mounting Google Drive...")
            drive.mount('/content/drive')
    except Exception:
        PLATFORM['ckpt_dir'] = '/content/phase_snn_ckpts'
        print("Drive not available — checkpoints saved locally")

os.makedirs(PLATFORM['ckpt_dir'], exist_ok=True)
print(f"Checkpoints: {PLATFORM['ckpt_dir']}")


def clip_grads(parameters, max_norm=1.0):
    """Standard gradient clipping — no complex dtype handling needed."""
    params = [p for p in parameters if p.grad is not None]
    if not params:
        return
    total = torch.cat([p.grad.detach().flatten() for p in params]).norm(2)
    clip  = max_norm / (total + 1e-6)
    if clip < 1.0:
        for p in params:
            if p.grad is not None:
                p.grad.detach().mul_(clip)


# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION — edit these
# ═══════════════════════════════════════════════════════════════════════════════

FORCE_FRESH_START  = True    # True = ignore old checkpoints (required for Phase 3)
LM_TOTAL_STEPS     = 30000
CKPT_EVERY         = 500
LM_BATCH           = 16      # per-GPU — safe for all platforms
GRAD_ACCUM         = 4       # effective batch = 64
SEQ_LEN            = 128
VOCAB_SAMPLE       = 10_000  # texts for vocab building
MAX_TRAIN_TOKENS   = 1_000_000
MAX_VAL_TOKENS     =   100_000
LM_LR              = 1e-3
WARMUP             = 1000

# ═══════════════════════════════════════════════════════════════════════════════
# RUN 1: DATA
# ═══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("RUN 1: DATA")
print("="*60)

from datasets import load_dataset
print("\nLoading WikiText-103...")

# WikiText-103 loading — tries Kaggle local files first, then HuggingFace
# Kaggle dataset: search "vadimkurochkin/wikitext-103" in + Add Data
# Kaggle Internet must be ON for HuggingFace fallback
import os

# Multiple known Kaggle slugs for WikiText-103
WT_KAGGLE_CANDIDATES = [
    '/kaggle/input/wikitext-103',
    '/kaggle/input/wikitext103',
    '/kaggle/input/wikitext-103-raw',
]

def _load_wikitext_local(path):
    """Load WikiText-103 from local Kaggle dataset files."""
    train, val = [], []
    # Try different possible filenames
    for train_name in ['wiki.train.tokens', 'wiki.train.raw',
                       'train.txt', 'wikitext-103/wiki.train.tokens']:
        fpath = os.path.join(path, train_name)
        if os.path.exists(fpath):
            with open(fpath, encoding='utf-8') as f:
                train = [l.strip() for l in f if l.strip()
                         and not l.startswith(' = ')]
            break
    for val_name in ['wiki.valid.tokens', 'wiki.valid.raw',
                     'valid.txt', 'wikitext-103/wiki.valid.tokens']:
        fpath = os.path.join(path, val_name)
        if os.path.exists(fpath):
            with open(fpath, encoding='utf-8') as f:
                val = [l.strip() for l in f if l.strip()
                       and not l.startswith(' = ')]
            break
    return train, val

train_texts, val_texts = [], []
for candidate in WT_KAGGLE_CANDIDATES:
    if os.path.exists(candidate):
        print(f"  Loading WikiText-103 from: {candidate}")
        # List what's inside so user can debug path issues
        try:
            files = os.listdir(candidate)[:6]
            print(f"  Files: {files}")
        except Exception:
            pass
        train_texts, val_texts = _load_wikitext_local(candidate)
        if train_texts:
            print(f"  Loaded {len(train_texts):,} train / {len(val_texts):,} val lines")
            break

if not train_texts:
    print("  WikiText-103 not found locally — downloading via HuggingFace")
    print("  (Kaggle: enable Internet in Notebook Settings)")
    wt103       = load_dataset('wikitext', 'wikitext-103-v1')
    train_texts = [x['text'] for x in wt103['train']      if x['text'].strip()]
    val_texts   = [x['text'] for x in wt103['validation'] if x['text'].strip()]

print(f"  {len(train_texts):,} train / {len(val_texts):,} val articles")

# Vocabulary
print(f"\nBuilding vocabulary ({VOCAB_SAMPLE:,} texts)...")
t0 = time.time()
word_freq = Counter()
for i, text in enumerate(train_texts[:VOCAB_SAMPLE]):
    word_freq.update(text.lower().split())
    if (i+1) % 2000 == 0:
        print(f"  {i+1:,}/{VOCAB_SAMPLE:,} texts, "
              f"{len(word_freq):,} unique words...")

token_to_id = {'<pad>':0, '<unk>':1, '<bos>':2, '<eos>':3}
for word, _ in word_freq.most_common(8192 - 10):
    if word not in token_to_id:
        token_to_id[word] = len(token_to_id)
for c in 'abcdefghijklmnopqrstuvwxyz0123456789.,!?;:()-':
    if c not in token_to_id:
        token_to_id[c] = len(token_to_id)

VOCAB_SIZE  = len(token_to_id)
id_to_token = {v: k for k, v in token_to_id.items()}
BOS_ID = token_to_id['<bos>']
EOS_ID = token_to_id['<eos>']
UNK_ID = token_to_id['<unk>']
print(f"  Vocab: {VOCAB_SIZE:,} tokens  ({time.time()-t0:.1f}s)")

# Coverage check
sample = sum((t.lower().split() for t in val_texts[:500]), [])
cov    = sum(1 for w in sample if w in token_to_id) / max(len(sample), 1)
unk_pct= 1 - cov
print(f"  Val coverage: {cov:.1%}  UNK rate: {unk_pct:.1%}")

def encode(text):
    ids = [BOS_ID]
    ids.extend(token_to_id.get(w, UNK_ID) for w in text.lower().split())
    ids.append(EOS_ID)
    return ids

def decode_tokens(ids):
    return ' '.join(id_to_token.get(i, '') for i in ids
                    if i not in (BOS_ID, EOS_ID, 0))

# Encode
print("\nEncoding corpus (with progress)...")
t0 = time.time()

def encode_corpus(texts, max_tokens, label):
    all_ids = []
    for i, text in enumerate(texts):
        if len(all_ids) >= max_tokens:
            break
        all_ids.extend(encode(text))
        if (i+1) % 5000 == 0:
            print(f"  {label}: {len(all_ids):,} / {max_tokens:,} tokens...")
    return np.array(all_ids[:max_tokens], dtype=np.int32)

train_tokens = encode_corpus(train_texts, MAX_TRAIN_TOKENS, 'train')
val_tokens   = encode_corpus(val_texts,   MAX_VAL_TOKENS,   'val')
print(f"  Train: {len(train_tokens):,}  Val: {len(val_tokens):,}  "
      f"({time.time()-t0:.1f}s)")

class TokenDataset(Dataset):
    def __init__(self, data, seq_len):
        self.data    = torch.tensor(data, dtype=torch.long)
        self.seq_len = seq_len
    def __len__(self):
        return max(0, len(self.data) - self.seq_len - 1)
    def __getitem__(self, idx):
        x = self.data[idx:idx+self.seq_len]
        y = self.data[idx+1:idx+self.seq_len+1]
        return x, y

train_dl = DataLoader(
    TokenDataset(train_tokens, SEQ_LEN),
    batch_size=LM_BATCH, shuffle=True,
    num_workers=PLATFORM['num_workers'],
    pin_memory=PLATFORM['pin_memory'],
    persistent_workers=False,
)
val_dl = DataLoader(
    TokenDataset(val_tokens, SEQ_LEN),
    batch_size=LM_BATCH, shuffle=False,
    num_workers=PLATFORM['num_workers'],
)

# ═══════════════════════════════════════════════════════════════════════════════
# RUN 2: MODEL
# ═══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("RUN 2: MODEL")
print("="*60)

lm = PhaseLM(
    vocab_size    = VOCAB_SIZE,
    D_embed       = 256,
    K_head        = 256,
    N_heads       = 8,
    N_layers      = 4,
    dropout       = 0.1,
    lam_sharp     = 0.02,
    ffn_expansion = 2,
    use_ffn       = True,
).to(DEVICE)

pc = lm.param_count()
print(f"\nPhase 3 model: K=2048  N_layers=4  FFN_exp=2  vocab={VOCAB_SIZE:,}")
print(f"  Parameters: {pc['total']:,}  ({pc['mb']:.1f}MB)")
print(f"  Real weights only — no complex dtype")

scan_params = sum(p.numel() for l in lm.scan_layers for p in l.parameters())
print(f"  Scan layer params: {scan_params:,} "
      f"({scan_params/pc['total']*100:.1f}% of model)")
print(f"  Phase 2 had: 16,388 (0.07%) — plateau at PPL 294")

# Verify VRAM before training
if DEVICE.type == 'cuda':
    torch.cuda.reset_peak_memory_stats()
    xb_t = torch.randint(0, VOCAB_SIZE, (LM_BATCH, SEQ_LEN), device=DEVICE)
    yb_t = torch.randint(0, VOCAB_SIZE, (LM_BATCH, SEQ_LEN), device=DEVICE)
    use_amp = True
    with torch.cuda.amp.autocast(enabled=use_amp):
        _, l_t = lm(xb_t, yb_t)
    l_t.backward()
    lm.zero_grad()
    peak = torch.cuda.max_memory_allocated() / 1e9
    torch.cuda.reset_peak_memory_stats()
    del xb_t, yb_t, l_t
    torch.cuda.empty_cache()
    vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"\n  VRAM check: {peak:.2f}GB peak / {vram_total:.1f}GB available")
    print(f"  {'✓ safe' if peak < vram_total * 0.7 else '⚠ tight — reduce batch'}")
    use_amp = (DEVICE.type == 'cuda')
else:
    use_amp = False
    print("\n  CPU mode — mixed precision disabled")

# ═══════════════════════════════════════════════════════════════════════════════
# RUN 3: TRAIN
# ═══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("RUN 3: TRAINING")
print("="*60)

ckpt_mgr      = CheckpointManager(PLATFORM['ckpt_dir'],
                                   prefix='lm_phase3', keep_last=3)
opt_lm        = optim.AdamW(lm.parameters(), lr=LM_LR,
                              weight_decay=1e-2, betas=(0.9, 0.95))
scaler        = torch.cuda.amp.GradScaler(enabled=use_amp)
start_step    = 1
loss_hist     = []
best_val_loss = float('inf')
best_state    = None

if not FORCE_FRESH_START:
    state = ckpt_mgr.load_latest()
    if state is not None:
        try:
            lm.load_state_dict(state['model'])
            opt_lm.load_state_dict(state['optimizer'])
            start_step    = state['step'] + 1
            loss_hist     = state['loss_hist']
            best_val_loss = state.get('best_val_loss', float('inf'))
            best_state    = state.get('best_state')
            lm.to(DEVICE)
            print(f"  Resumed from step {state['step']}")
        except Exception as e:
            print(f"  Checkpoint incompatible ({e}) — starting fresh")
            start_step = 1
else:
    print("  FORCE_FRESH_START=True — fresh training")

remaining = LM_TOTAL_STEPS - start_step + 1
if remaining <= 0:
    print(f"\n  Done. Increase LM_TOTAL_STEPS or set FORCE_FRESH_START=True.")
else:
    LM_CONFIG = dict(vocab=VOCAB_SIZE, K=2048, layers=4,
                     total_steps=LM_TOTAL_STEPS)
    train_iter = iter(train_dl)
    opt_lm.zero_grad()
    t0 = time.time()

    print(f"\nSteps {start_step}→{LM_TOTAL_STEPS}  "
          f"eff_batch={LM_BATCH*GRAD_ACCUM}  seq={SEQ_LEN}  "
          f"warmup={WARMUP}  amp={use_amp}")
    print(f"\n{'step':>6}  {'loss':>8}  {'ppl':>8}  "
          f"{'val_loss':>9}  {'val_ppl':>8}  {'t':>6}")
    print("-"*56)

    accum = 0.0
    for step in range(start_step, LM_TOTAL_STEPS + 1):
        lm.train()
        cosine_lr_schedule(opt_lm, step, LM_TOTAL_STEPS,
                           LM_LR, lr_min=1e-5, warmup_steps=WARMUP)

        for _ in range(GRAD_ACCUM):
            try:
                xb, yb = next(train_iter)
            except StopIteration:
                train_iter = iter(train_dl)
                xb, yb = next(train_iter)
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            with torch.cuda.amp.autocast(enabled=use_amp):
                _, loss = lm(xb, yb)
            scaler.scale(loss / GRAD_ACCUM).backward()
            accum += loss.item() / GRAD_ACCUM

        scaler.unscale_(opt_lm)
        clip_grads(lm.parameters())
        scaler.step(opt_lm)
        scaler.update()
        opt_lm.zero_grad()
        loss_hist.append(accum)
        accum = 0.0

        if step % CKPT_EVERY == 0 or step == LM_TOTAL_STEPS:
            lm.eval()
            vl_list = []
            with torch.no_grad():
                for i, (xv, yv) in enumerate(val_dl):
                    if i >= 50: break
                    xv, yv = xv.to(DEVICE), yv.to(DEVICE)
                    with torch.cuda.amp.autocast(enabled=use_amp):
                        _, vl = lm(xv, yv)
                    vl_list.append(vl.item())
            val_loss = float(np.mean(vl_list))
            val_ppl  = math.exp(min(val_loss, 20))
            tr_loss  = float(np.mean(loss_hist[-min(100, len(loss_hist)):]))
            tr_ppl   = math.exp(min(tr_loss, 20))

            improved = val_loss < best_val_loss
            if improved:
                best_val_loss = val_loss
                best_state    = {k: v.clone()
                                 for k, v in lm.state_dict().items()}

            mem = ""
            if DEVICE.type == 'cuda' and step <= CKPT_EVERY * 2:
                gb = torch.cuda.max_memory_allocated() / 1e9
                torch.cuda.reset_peak_memory_stats()
                mem = f"  [{gb:.1f}GB]"

            print(f"{step:>6}  {tr_loss:>8.4f}  {tr_ppl:>8.1f}"
                  f"  {val_loss:>9.4f}  {val_ppl:>8.1f}"
                  f"  {time.time()-t0:>5.0f}s"
                  f"{'  ★' if improved else ''}{mem}")

            ckpt_mgr.save(step, lm, opt_lm, loss_hist, LM_CONFIG,
                          extra={'best_val_loss': best_val_loss,
                                 'best_state': best_state})

    print(f"\n  Best val PPL: {math.exp(min(best_val_loss,20)):.1f}")
    print(f"  Phase 2 best: 294.0")
    print(f"  {'✓ Improvement' if best_val_loss < math.log(294) else '~ Still training'}")

# ═══════════════════════════════════════════════════════════════════════════════
# RUN 4: GENERATION
# ═══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("RUN 4: GENERATION")
print("="*60)

if best_state is not None:
    lm.load_state_dict(best_state)
lm.eval()

for prompt in ["the history of", "scientists discovered that",
               "the government announced", "in recent years"]:
    ids  = torch.tensor([encode(prompt)], dtype=torch.long).to(DEVICE)
    out  = lm.generate(ids, max_new=60, temperature=0.8, top_k=50)
    print(f"\n  '{prompt}'")
    print(f"  → '{decode_tokens(out[0].cpu().tolist())}'")

print(f"\n{'='*60}")
print("PHASE 3 COMPLETE")
print(f"{'='*60}")
final_ppl = math.exp(min(best_val_loss, 20))
print(f"  Best val PPL: {final_ppl:.1f}  (Phase 2: 294)")
print(f"  Model: {pc['total']:,} params  ({pc['mb']:.1f}MB)")
print(f"  Platform: {PLATFORM['name']}")


## Cell 4: Export for Rust

In [ ]:
try:
    import os
    ckpt = PLATFORM['ckpt_dir']
    export_model(lm, token_to_id, os.path.join(ckpt, 'weights.json'))
    print('Weights exported')
except NameError as e:
    print(f'Run Cell 3 first: {e}')

RUST_FILES = {
    'Cargo.toml':  base64.b64decode('W3BhY2thZ2VdCm5hbWUgPSAicGhhc2Vfc25uIgp2ZXJzaW9uID0gIjAuMS4wIgplZGl0aW9uID0gIjIwMjEiCmRlc2NyaXB0aW9uID0gIlBoYXNlLVNOTiBpbmZlcmVuY2UgZW5naW5lIOKAlCBDUFUsIG5vIFB5dGhvbiBydW50aW1lIgoKW2RlcGVuZGVuY2llc10KbmRhcnJheSA9IHsgdmVyc2lvbiA9ICIwLjE1IiwgZmVhdHVyZXMgPSBbInJheW9uIl0gfQpudW0tY29tcGxleCA9ICIwLjQiCnJheW9uID0gIjEuOCIKc2VyZGUgPSB7IHZlcnNpb24gPSAiMS4wIiwgZmVhdHVyZXMgPSBbImRlcml2ZSJdIH0Kc2VyZGVfanNvbiA9ICIxLjAiCgpbcHJvZmlsZS5yZWxlYXNlXQpvcHQtbGV2ZWwgPSAzCmx0byA9IHRydWUKY29kZWdlbi11bml0cyA9IDEKCltbYmluXV0KbmFtZSA9ICJwaGFzZV9zbm5faW5mZXJlbmNlIgpwYXRoID0gInNyYy9tYWluLnJzIgo=').decode(),
    'src/lib.rs':  base64.b64decode('Ly8hIFBoYXNlLVNOTiBJbmZlcmVuY2UgRW5naW5lCi8vIQovLyEgQ1BVIGluZmVyZW5jZSBmb3IgdGhlIFBoYXNlLVNOTiBhcmNoaXRlY3R1cmUuCi8vISBMb2FkcyB3ZWlnaHRzIGV4cG9ydGVkIGZyb20gUHlUb3JjaCB0cmFpbmluZy4KLy8hCi8vISBBcmNoaXRlY3R1cmU6Ci8vISAgIE11bHRpSGVhZFBoYXNlRW5jb2RlciDihpIgUGhhc2VTY2FuTGF5ZXIgw5cgTiDihpIgTE0gaGVhZAovLyEKLy8hIFVzYWdlOgovLyEgICBsZXQgbW9kZWwgPSBQaGFzZUxNOjpsb2FkKCJ3ZWlnaHRzLmpzb24iKS51bndyYXAoKTsKLy8hICAgbGV0IHRva2VucyA9IHZlYyFbMnUzMiwgMTAsIDQyLCA3XTsgIC8vIDxib3M+ICsgdG9rZW4gaWRzCi8vISAgIGxldCBuZXh0X2xvZ2l0cyA9IG1vZGVsLmZvcndhcmQoJnRva2Vucyk7Ci8vISAgIGxldCBuZXh0X3Rva2VuID0gbmV4dF9sb2dpdHMuYXJnbWF4KCk7Cgp1c2UgbmRhcnJheTo6e0FycmF5MSwgQXJyYXkyLCBBcnJheVZpZXcxLCBBeGlzLCBzfTsKdXNlIG51bV9jb21wbGV4OjpDb21wbGV4MzI7CnVzZSBzZXJkZTo6e0Rlc2VyaWFsaXplLCBTZXJpYWxpemV9Owp1c2Ugc3RkOjpmMzI6OmNvbnN0czo6UEk7CgovLyDilIDilIAgV2VpZ2h0IHN0cnVjdHVyZXMgKGxvYWRlZCBmcm9tIEpTT04gZXhwb3J0KSDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKCiNbZGVyaXZlKFNlcmlhbGl6ZSwgRGVzZXJpYWxpemUpXQpwdWIgc3RydWN0IFBoYXNlRW5jb2RlckhlYWRXZWlnaHRzIHsKICAgIC8vLyBXOiBzaGFwZSBbSywgRF0gYXMgZmxhdCB2ZWMgb2YgKHJlYWwsIGltYWcpIHBhaXJzCiAgICBwdWIgd19yZWFsOiBWZWM8ZjMyPiwKICAgIHB1YiB3X2ltYWc6IFZlYzxmMzI+LAogICAgcHViIG9tZWdhOiAgVmVjPGYzMj4sCiAgICBwdWIgazogdXNpemUsCiAgICBwdWIgZDogdXNpemUsCn0KCiNbZGVyaXZlKFNlcmlhbGl6ZSwgRGVzZXJpYWxpemUpXQpwdWIgc3RydWN0IFBoYXNlU2NhbkxheWVyV2VpZ2h0cyB7CiAgICBwdWIgYWxwaGE6ICAgICAgZjMyLAogICAgLy8vIFBoYXNlTm9ybSBnYW1tYSBhbmQgYmV0YQogICAgcHViIG5vcm1fZ2FtbWE6IFZlYzxmMzI+LAogICAgcHViIG5vcm1fYmV0YTogIFZlYzxmMzI+LAp9CgojW2Rlcml2ZShTZXJpYWxpemUsIERlc2VyaWFsaXplKV0KcHViIHN0cnVjdCBNb2RlbFdlaWdodHMgewogICAgcHViIHZvY2FiX3NpemU6IHVzaXplLAogICAgcHViIGRfZW1iZWQ6ICAgIHVzaXplLAogICAgcHViIGtfdG90YWw6ICAgIHVzaXplLAogICAgcHViIG5fbGF5ZXJzOiAgIHVzaXplLAogICAgLy8vIFRva2VuIGVtYmVkZGluZyB0YWJsZTogW3ZvY2FiX3NpemUsIGRfZW1iZWRdCiAgICBwdWIgZW1iZWRkaW5nOiAgVmVjPGYzMj4sCiAgICBwdWIgaGVhZHM6ICAgICAgVmVjPFBoYXNlRW5jb2RlckhlYWRXZWlnaHRzPiwKICAgIHB1YiBzY2FuX2xheWVyczogVmVjPFBoYXNlU2NhbkxheWVyV2VpZ2h0cz4sCiAgICAvLy8gTE0gaGVhZDogW3ZvY2FiX3NpemUsIGtfdG90YWxdCiAgICBwdWIgbG1faGVhZF93OiAgVmVjPGYzMj4sCiAgICBwdWIgbG1faGVhZF9iOiAgVmVjPGYzMj4sCiAgICAvLy8gVm9jYWJ1bGFyeTogdG9rZW5faWQg4oaSIHN0cmluZwogICAgcHViIHZvY2FiOiAgICAgIHN0ZDo6Y29sbGVjdGlvbnM6Okhhc2hNYXA8dTMyLCBTdHJpbmc+LAp9CgovLyDilIDilIAgQ29yZSBvcGVyYXRpb25zIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAoKLy8vIFBoYXNlIGNvc2luZSBzaW1pbGFyaXR5IGJldHdlZW4gdHdvIHZlY3RvcnMKI1tpbmxpbmVdCnB1YiBmbiBwaGFzZV9jb3Nfc2ltKGE6ICZbZjMyXSwgYjogJltmMzJdKSAtPiBmMzIgewogICAgYXNzZXJ0X2VxIShhLmxlbigpLCBiLmxlbigpKTsKICAgIGxldCBrID0gYS5sZW4oKSBhcyBmMzI7CiAgICBhLml0ZXIoKS56aXAoYi5pdGVyKCkpCiAgICAgICAgLm1hcCh8KCZhaSwgJmJpKXwgKGFpIC0gYmkpLmNvcygpKQogICAgICAgIC5zdW06OjxmMzI+KCkgLyBrCn0KCi8vLyBFbmNvZGUgYSBzaW5nbGUgZW1iZWRkaW5nIHRocm91Z2ggb25lIHBoYXNlIGhlYWQKLy8vIEU6IFtEXSwgVzogW0ssIERdIGNvbXBsZXgsIG9tZWdhOiBbS10g4oaSIHBoaTogW0tdCnB1YiBmbiBlbmNvZGVfaGVhZCgKICAgIGU6ICAgICAmW2YzMl0sCiAgICB3X3JlYWw6ICZbZjMyXSwKICAgIHdfaW1hZzogJltmMzJdLAogICAgb21lZ2E6ICAmW2YzMl0sCiAgICBrOiB1c2l6ZSwgZDogdXNpemUsCikgLT4gVmVjPGYzMj4gewogICAgbGV0IG11dCBwaGkgPSB2ZWMhWzAuMGYzMjsga107CiAgICBmb3Iga2kgaW4gMC4uayB7CiAgICAgICAgLy8geiA9IHN1bV9kIEVbZF0gKiBjb25qKFdba2ksIGRdKQogICAgICAgIC8vIGNvbmooVykgPSAod19yZWFsIC0gaSp3X2ltYWcpCiAgICAgICAgbGV0IG11dCB6X3JlYWwgPSAwLjBmMzI7CiAgICAgICAgbGV0IG11dCB6X2ltYWcgPSAwLjBmMzI7CiAgICAgICAgZm9yIGRpIGluIDAuLmQgewogICAgICAgICAgICBsZXQgZV9kID0gZVtkaV07CiAgICAgICAgICAgIHpfcmVhbCArPSBlX2QgKiB3X3JlYWxba2kgKiBkICsgZGldOwogICAgICAgICAgICB6X2ltYWcgKz0gZV9kICogd19pbWFnW2tpICogZCArIGRpXTsgIC8vIGNvbmogZmxpcHMgc2lnbiBvZiBpbWFnCiAgICAgICAgfQogICAgICAgIGxldCB6ID0gQ29tcGxleDMyOjpuZXcoel9yZWFsLCB6X2ltYWcpOwogICAgICAgIGxldCBtYWcgID0gei5ub3JtKCkgKyAxZS0xMjsKICAgICAgICBsZXQgZ2F0ZSA9IG1hZy50YW5oKCk7CiAgICAgICAgbGV0IGFuZ2xlID0gei5hcmcoKTsKICAgICAgICBwaGlba2ldID0gYW5nbGUgKiBvbWVnYVtraV0gKiBnYXRlOwogICAgfQogICAgcGhpCn0KCi8vLyBIaWxsaXMtU3RlZWxlIGNhdXNhbCBwcmVmaXggc2NhbiBvbiBhIHNlcXVlbmNlCi8vLyBpbnB1dDogW0wsIEtdIOKGkiBvdXRwdXQ6IFtMLCBLXSB3aGVyZSBvdXRwdXRbdF0gPSBzdW0oaW5wdXRbMC4uPXRdKQpwdWIgZm4gaGlsbGlzX3N0ZWVsZV9zY2FuKHg6ICZbVmVjPGYzMj5dLCBrOiB1c2l6ZSkgLT4gVmVjPFZlYzxmMzI+PiB7CiAgICBsZXQgbCA9IHgubGVuKCk7CiAgICBpZiBsID09IDAgeyByZXR1cm4gdmVjIVtdOyB9CiAgICBsZXQgbXV0IHJlczogVmVjPFZlYzxmMzI+PiA9IHgudG9fdmVjKCk7CiAgICBsZXQgbnVtX3N0ZXBzID0gKGwgYXMgZjMyKS5sb2cyKCkuY2VpbCgpIGFzIHVzaXplICsgMTsKICAgIGZvciBpIGluIDAuLm51bV9zdGVwcyB7CiAgICAgICAgbGV0IHN0cmlkZSA9IDF1c2l6ZSA8PCBpOwogICAgICAgIGlmIHN0cmlkZSA+PSBsIHsgYnJlYWs7IH0KICAgICAgICBsZXQgcHJldiA9IHJlcy5jbG9uZSgpOwogICAgICAgIGZvciB0IGluIHN0cmlkZS4ubCB7CiAgICAgICAgICAgIGZvciBraSBpbiAwLi5rIHsKICAgICAgICAgICAgICAgIHJlc1t0XVtraV0gKz0gcHJldlt0IC0gc3RyaWRlXVtraV07CiAgICAgICAgICAgIH0KICAgICAgICB9CiAgICB9CiAgICByZXMKfQoKLy8vIFBoYXNlTm9ybTogbGF5ZXIgbm9ybSBhcHBsaWVkIHRvIGNvcyB0aGVuIHNpbiBzZXBhcmF0ZWx5CnB1YiBmbiBwaGFzZV9ub3JtKHBoaTogJltmMzJdLCBnYW1tYTogJltmMzJdLCBiZXRhOiAmW2YzMl0pIC0+IFZlYzxmMzI+IHsKICAgIGxldCBrID0gcGhpLmxlbigpOwogICAgbGV0IGNvc19waGk6IFZlYzxmMzI+ID0gcGhpLml0ZXIoKS5tYXAofCZwfCBwLmNvcygpKS5jb2xsZWN0KCk7CiAgICBsZXQgc2luX3BoaTogVmVjPGYzMj4gPSBwaGkuaXRlcigpLm1hcCh8JnB8IHAuc2luKCkpLmNvbGxlY3QoKTsKCiAgICBmbiBsYXllcl9ub3JtXzFkKHg6ICZbZjMyXSkgLT4gVmVjPGYzMj4gewogICAgICAgIGxldCBtZWFuID0geC5pdGVyKCkuc3VtOjo8ZjMyPigpIC8geC5sZW4oKSBhcyBmMzI7CiAgICAgICAgbGV0IHZhciAgPSB4Lml0ZXIoKS5tYXAofCZ4aXwgKHhpIC0gbWVhbikucG93aSgyKSkuc3VtOjo8ZjMyPigpCiAgICAgICAgICAgICAgICAgICAvIHgubGVuKCkgYXMgZjMyOwogICAgICAgIGxldCBzdGQgID0gKHZhciArIDFlLTUpLnNxcnQoKTsKICAgICAgICB4Lml0ZXIoKS5tYXAofCZ4aXwgKHhpIC0gbWVhbikgLyBzdGQpLmNvbGxlY3QoKQogICAgfQoKICAgIGxldCBjb3NfbiA9IGxheWVyX25vcm1fMWQoJmNvc19waGkpOwogICAgbGV0IHNpbl9uID0gbGF5ZXJfbm9ybV8xZCgmc2luX3BoaSk7CgogICAgLy8gUmVjb25zdHJ1Y3Q6IGF0YW4yKHNpbl9ub3JtLCBjb3Nfbm9ybSkgKiBnYW1tYSArIGJldGEKICAgICgwLi5rKS5tYXAofGl8IHsKICAgICAgICBzaW5fbltpXS5hdGFuMihjb3NfbltpXSkgKiBnYW1tYVtpXSArIGJldGFbaV0KICAgIH0pLmNvbGxlY3QoKQp9CgovLy8gT25lIFBoYXNlU2NhbkxheWVyIGZvcndhcmQgcGFzcyB3aXRoIHJlc2lkdWFsIGNvbm5lY3Rpb24KcHViIGZuIHNjYW5fbGF5ZXJfZm9yd2FyZCgKICAgIHBoaTogICAgJltmMzJdLAogICAgY3R4OiAgICAmW2YzMl0sCiAgICBhbHBoYTogIGYzMiwKICAgIGdhbW1hOiAgJltmMzJdLAogICAgYmV0YTogICAmW2YzMl0sCikgLT4gVmVjPGYzMj4gewogICAgbGV0IGsgPSBwaGkubGVuKCk7CiAgICBsZXQgYWxwaGFfcyA9IDEuMCAvICgxLjAgKyAoLWFscGhhKS5leHAoKSk7ICAvLyBzaWdtb2lkCiAgICAvLyBNaXg6IHBoaSArIGFscGhhICogKGN0eCAtIHBoaSkKICAgIGxldCBtaXhlZDogVmVjPGYzMj4gPSAoMC4uaykubWFwKHxpfCB7CiAgICAgICAgcGhpW2ldICsgYWxwaGFfcyAqIChjdHhbaV0gLSBwaGlbaV0pCiAgICB9KS5jb2xsZWN0KCk7CiAgICAvLyBOb3JtYWxpc2UKICAgIGxldCBub3JtZWQgPSBwaGFzZV9ub3JtKCZtaXhlZCwgZ2FtbWEsIGJldGEpOwogICAgLy8gUmVzaWR1YWw6IHBoaSArIG5vcm1lZAogICAgKDAuLmspLm1hcCh8aXwgcGhpW2ldICsgbm9ybWVkW2ldKS5jb2xsZWN0KCkKfQoKLy8g4pSA4pSAIEZ1bGwgbW9kZWwg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACgpwdWIgc3RydWN0IFBoYXNlTE0gewogICAgd2VpZ2h0czogTW9kZWxXZWlnaHRzLAp9CgppbXBsIFBoYXNlTE0gewogICAgcHViIGZuIGxvYWQocGF0aDogJnN0cikgLT4gUmVzdWx0PFNlbGYsIEJveDxkeW4gc3RkOjplcnJvcjo6RXJyb3I+PiB7CiAgICAgICAgbGV0IGpzb24gPSBzdGQ6OmZzOjpyZWFkX3RvX3N0cmluZyhwYXRoKT87CiAgICAgICAgbGV0IHdlaWdodHM6IE1vZGVsV2VpZ2h0cyA9IHNlcmRlX2pzb246OmZyb21fc3RyKCZqc29uKT87CiAgICAgICAgcHJpbnRsbiEoIkxvYWRlZCBQaGFzZS1TTk46IHZvY2FiPXt9IEs9e30gbGF5ZXJzPXt9IiwKICAgICAgICAgICAgICAgICB3ZWlnaHRzLnZvY2FiX3NpemUsIHdlaWdodHMua190b3RhbCwgd2VpZ2h0cy5uX2xheWVycyk7CiAgICAgICAgT2soU2VsZiB7IHdlaWdodHMgfSkKICAgIH0KCiAgICAvLy8gRm9yd2FyZCBwYXNzOiB0b2tlbiBpZHMg4oaSIGxvZ2l0cyBvdmVyIHZvY2FidWxhcnkKICAgIC8vLyB0b2tlbnM6IHNlcXVlbmNlIG9mIHRva2VuIGlkcyAoaW5jbHVkaW5nIDxib3M+KQogICAgcHViIGZuIGZvcndhcmQoJnNlbGYsIHRva2VuczogJlt1MzJdKSAtPiBWZWM8ZjMyPiB7CiAgICAgICAgbGV0IHcgID0gJnNlbGYud2VpZ2h0czsKICAgICAgICBsZXQgbCAgPSB0b2tlbnMubGVuKCk7CiAgICAgICAgbGV0IGQgID0gdy5kX2VtYmVkOwogICAgICAgIGxldCBrICA9IHcua190b3RhbDsKICAgICAgICBsZXQgbl9oZWFkcyA9IHcuaGVhZHMubGVuKCk7CiAgICAgICAgbGV0IGtfaGVhZCAgPSBrIC8gbl9oZWFkczsKCiAgICAgICAgLy8gU3RlcCAxOiBlbWJlZCB0b2tlbnMg4oaSIFtMLCBEXQogICAgICAgIGxldCBlbWJlZGRpbmdzOiBWZWM8VmVjPGYzMj4+ID0gdG9rZW5zLml0ZXIoKS5tYXAofCZ0b2t8IHsKICAgICAgICAgICAgbGV0IGlkeCA9ICh0b2sgYXMgdXNpemUpLm1pbih3LnZvY2FiX3NpemUgLSAxKTsKICAgICAgICAgICAgdy5lbWJlZGRpbmdbaWR4KmQuLihpZHgrMSkqZF0udG9fdmVjKCkKICAgICAgICB9KS5jb2xsZWN0KCk7CgogICAgICAgIC8vIFN0ZXAgMjogbXVsdGktaGVhZCBwaGFzZSBlbmNvZGUg4oaSIFtMLCBLXQogICAgICAgIGxldCBtdXQgcGhpX3NlcTogVmVjPFZlYzxmMzI+PiA9IGVtYmVkZGluZ3MuaXRlcigpLm1hcCh8ZXwgewogICAgICAgICAgICBsZXQgbXV0IHBoaV9hbGwgPSBWZWM6OndpdGhfY2FwYWNpdHkoayk7CiAgICAgICAgICAgIGZvciBoZWFkIGluICZ3LmhlYWRzIHsKICAgICAgICAgICAgICAgIGxldCBwaGlfaCA9IGVuY29kZV9oZWFkKGUsICZoZWFkLndfcmVhbCwgJmhlYWQud19pbWFnLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgJmhlYWQub21lZ2EsIGhlYWQuaywgaGVhZC5kKTsKICAgICAgICAgICAgICAgIHBoaV9hbGwuZXh0ZW5kKHBoaV9oKTsKICAgICAgICAgICAgfQogICAgICAgICAgICBwaGlfYWxsCiAgICAgICAgfSkuY29sbGVjdCgpOwoKICAgICAgICAvLyBTdGVwIDM6IHNjYW4gbGF5ZXJzIHdpdGggcmVzaWR1YWxzCiAgICAgICAgZm9yIGxheWVyIGluICZ3LnNjYW5fbGF5ZXJzIHsKICAgICAgICAgICAgbGV0IGN0eCA9IGhpbGxpc19zdGVlbGVfc2NhbigmcGhpX3NlcSwgayk7CiAgICAgICAgICAgIHBoaV9zZXEgPSBwaGlfc2VxLml0ZXIoKS56aXAoY3R4Lml0ZXIoKSkubWFwKHwocGhpLCBjKXwgewogICAgICAgICAgICAgICAgc2Nhbl9sYXllcl9mb3J3YXJkKHBoaSwgYywgbGF5ZXIuYWxwaGEsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgJmxheWVyLm5vcm1fZ2FtbWEsICZsYXllci5ub3JtX2JldGEpCiAgICAgICAgICAgIH0pLmNvbGxlY3QoKTsKICAgICAgICB9CgogICAgICAgIC8vIFN0ZXAgNDogTE0gaGVhZCBvbiBsYXN0IHBvc2l0aW9uIOKGkiBbdm9jYWJfc2l6ZV0KICAgICAgICBsZXQgbGFzdF9waGkgPSAmcGhpX3NlcVtsIC0gMV07CiAgICAgICAgbGV0IHZvY2FiICAgID0gdy52b2NhYl9zaXplOwogICAgICAgICgwLi52b2NhYikubWFwKHx2fCB7CiAgICAgICAgICAgIGxldCB3X3JvdyA9ICZ3LmxtX2hlYWRfd1t2KmsuLih2KzEpKmtdOwogICAgICAgICAgICBsZXQgZG90OiBmMzIgPSBsYXN0X3BoaS5pdGVyKCkuemlwKHdfcm93Lml0ZXIoKSkKICAgICAgICAgICAgICAgIC5tYXAofCgmcCwgJnd3KXwgcCAqIHd3KS5zdW0oKTsKICAgICAgICAgICAgZG90ICsgdy5sbV9oZWFkX2Jbdl0KICAgICAgICB9KS5jb2xsZWN0KCkKICAgIH0KCiAgICAvLy8gR3JlZWR5IG5leHQtdG9rZW4gcHJlZGljdGlvbgogICAgcHViIGZuIHByZWRpY3RfbmV4dCgmc2VsZiwgdG9rZW5zOiAmW3UzMl0pIC0+IHUzMiB7CiAgICAgICAgbGV0IGxvZ2l0cyA9IHNlbGYuZm9yd2FyZCh0b2tlbnMpOwogICAgICAgIGxvZ2l0cy5pdGVyKCkuZW51bWVyYXRlKCkKICAgICAgICAgICAgLm1heF9ieSh8KF8sIGEpLCAoXywgYil8IGEucGFydGlhbF9jbXAoYikudW53cmFwKCkpCiAgICAgICAgICAgIC5tYXAofChpLCBfKXwgaSBhcyB1MzIpCiAgICAgICAgICAgIC51bndyYXBfb3IoMSkKICAgIH0KCiAgICAvLy8gVGVtcGVyYXR1cmUtc2FtcGxlZCBnZW5lcmF0aW9uCiAgICBwdWIgZm4gZ2VuZXJhdGUoJnNlbGYsIHByb21wdDogJlt1MzJdLCBtYXhfbmV3OiB1c2l6ZSwKICAgICAgICAgICAgICAgICAgICB0ZW1wZXJhdHVyZTogZjMyKSAtPiBWZWM8dTMyPiB7CiAgICAgICAgbGV0IG11dCB0b2tlbnMgPSBwcm9tcHQudG9fdmVjKCk7CiAgICAgICAgZm9yIF8gaW4gMC4ubWF4X25ldyB7CiAgICAgICAgICAgIGxldCBsb2dpdHMgPSBzZWxmLmZvcndhcmQoJnRva2Vucyk7CiAgICAgICAgICAgIC8vIFRlbXBlcmF0dXJlIHNjYWxpbmcgKyBzb2Z0bWF4CiAgICAgICAgICAgIGxldCBtYXhfbCAgPSBsb2dpdHMuaXRlcigpLmNsb25lZCgpLmZvbGQoZjMyOjpORUdfSU5GSU5JVFksIGYzMjo6bWF4KTsKICAgICAgICAgICAgbGV0IHNjYWxlZDogVmVjPGYzMj4gPSBsb2dpdHMuaXRlcigpCiAgICAgICAgICAgICAgICAubWFwKHwmbHwgKChsIC0gbWF4X2wpIC8gdGVtcGVyYXR1cmUpLmV4cCgpKS5jb2xsZWN0KCk7CiAgICAgICAgICAgIGxldCBzdW06IGYzMiA9IHNjYWxlZC5pdGVyKCkuc3VtKCk7CiAgICAgICAgICAgIGxldCBwcm9iczogVmVjPGYzMj4gPSBzY2FsZWQuaXRlcigpLm1hcCh8JnN8IHMgLyBzdW0pLmNvbGxlY3QoKTsKICAgICAgICAgICAgLy8gU2FtcGxlCiAgICAgICAgICAgIGxldCByOiBmMzIgPSByYW5kX2YzMigpOwogICAgICAgICAgICBsZXQgbXV0IGN1bXN1bSA9IDAuMGYzMjsKICAgICAgICAgICAgbGV0IG11dCBuZXh0ID0gMXUzMjsgIC8vIDx1bms+IGZhbGxiYWNrCiAgICAgICAgICAgIGZvciAoaSwgJnApIGluIHByb2JzLml0ZXIoKS5lbnVtZXJhdGUoKSB7CiAgICAgICAgICAgICAgICBjdW1zdW0gKz0gcDsKICAgICAgICAgICAgICAgIGlmIGN1bXN1bSA+PSByIHsKICAgICAgICAgICAgICAgICAgICBuZXh0ID0gaSBhcyB1MzI7CiAgICAgICAgICAgICAgICAgICAgYnJlYWs7CiAgICAgICAgICAgICAgICB9CiAgICAgICAgICAgIH0KICAgICAgICAgICAgdG9rZW5zLnB1c2gobmV4dCk7CiAgICAgICAgfQogICAgICAgIHRva2VucwogICAgfQp9CgovLyBTaW1wbGUgTENHIHJhbmRvbSBmb3Igbm8tZGVwZW5kZW5jeSBzYW1wbGluZwpzdGF0aWMgbXV0IFJBTkRfU1RBVEU6IHU2NCA9IDEyMzQ1OwpmbiByYW5kX2YzMigpIC0+IGYzMiB7CiAgICB1bnNhZmUgewogICAgICAgIFJBTkRfU1RBVEUgPSBSQU5EX1NUQVRFLndyYXBwaW5nX211bCg2MzY0MTM2MjIzODQ2NzkzMDA1KQogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgLndyYXBwaW5nX2FkZCgxNDQyNjk1MDQwODg4OTYzNDA3KTsKICAgICAgICAoKFJBTkRfU1RBVEUgPj4gMzMpIGFzIGYzMikgLyAodTMyOjpNQVggYXMgZjMyKQogICAgfQp9Cg==').decode(),
    'src/main.rs': base64.b64decode('Ly8hIFBoYXNlLVNOTiBpbmZlcmVuY2UgQ0xJCi8vIQovLyEgVXNhZ2U6Ci8vISAgIHBoYXNlX3Nubl9pbmZlcmVuY2UgLS13ZWlnaHRzIHdlaWdodHMuanNvbiAtLXByb21wdCAiVGhlIGhpc3Rvcnkgb2YiCi8vISAgIHBoYXNlX3Nubl9pbmZlcmVuY2UgLS13ZWlnaHRzIHdlaWdodHMuanNvbiAtLWludGVyYWN0aXZlCgp1c2UgcGhhc2Vfc25uOjpQaGFzZUxNOwp1c2Ugc3RkOjppbzo6e3NlbGYsIEJ1ZlJlYWQsIFdyaXRlfTsKCmZuIG1haW4oKSB7CiAgICBsZXQgYXJnczogVmVjPFN0cmluZz4gPSBzdGQ6OmVudjo6YXJncygpLmNvbGxlY3QoKTsKCiAgICBsZXQgd2VpZ2h0c19wYXRoID0gYXJncy5pdGVyKCkucG9zaXRpb24ofGF8IGEgPT0gIi0td2VpZ2h0cyIpCiAgICAgICAgLmFuZF90aGVuKHxpfCBhcmdzLmdldChpKzEpKQogICAgICAgIC5tYXAofHN8IHMuYXNfc3RyKCkpCiAgICAgICAgLnVud3JhcF9vcigid2VpZ2h0cy5qc29uIik7CgogICAgbGV0IG1vZGVsID0gbWF0Y2ggUGhhc2VMTTo6bG9hZCh3ZWlnaHRzX3BhdGgpIHsKICAgICAgICBPayhtKSAgPT4gbSwKICAgICAgICBFcnIoZSkgPT4geyBlcHJpbnRsbiEoIkZhaWxlZCB0byBsb2FkOiB7fSIsIGUpOyByZXR1cm47IH0KICAgIH07CgogICAgaWYgYXJncy5jb250YWlucygmIi0taW50ZXJhY3RpdmUiLnRvX3N0cmluZygpKSB7CiAgICAgICAgcHJpbnRsbiEoIlBoYXNlLVNOTiBpbnRlcmFjdGl2ZSBtb2RlLiBUeXBlIGEgcHJvbXB0LCBFbnRlciB0byBnZW5lcmF0ZS4iKTsKICAgICAgICBsZXQgc3RkaW4gPSBpbzo6c3RkaW4oKTsKICAgICAgICBsb29wIHsKICAgICAgICAgICAgcHJpbnQhKCI+ICIpOwogICAgICAgICAgICBpbzo6c3Rkb3V0KCkuZmx1c2goKS51bndyYXAoKTsKICAgICAgICAgICAgbGV0IG11dCBsaW5lID0gU3RyaW5nOjpuZXcoKTsKICAgICAgICAgICAgaWYgc3RkaW4ubG9jaygpLnJlYWRfbGluZSgmbXV0IGxpbmUpLmlzX2VycigpIHsgYnJlYWs7IH0KICAgICAgICAgICAgbGV0IGxpbmUgPSBsaW5lLnRyaW0oKTsKICAgICAgICAgICAgaWYgbGluZS5pc19lbXB0eSgpIHsgY29udGludWU7IH0KICAgICAgICAgICAgaWYgbGluZSA9PSAicXVpdCIgeyBicmVhazsgfQogICAgICAgICAgICAvLyBTaW1wbGUgd29yZCB0b2tlbmlzYXRpb24gZm9yIGRlbW8KICAgICAgICAgICAgLy8gUmVhbCBkZXBsb3ltZW50IHdvdWxkIHVzZSB0aGUgc2F2ZWQgdm9jYWIKICAgICAgICAgICAgbGV0IHRva2VuczogVmVjPHUzMj4gPSB2ZWMhWzJdOyAgLy8gPGJvcz4KICAgICAgICAgICAgbGV0IGdlbmVyYXRlZCA9IG1vZGVsLmdlbmVyYXRlKCZ0b2tlbnMsIDUwLCAwLjgpOwogICAgICAgICAgICBwcmludGxuISgiR2VuZXJhdGVkOiB7Oj99IiwgZ2VuZXJhdGVkKTsKICAgICAgICB9CiAgICB9IGVsc2UgewogICAgICAgIC8vIEJlbmNobWFyayBtb2RlCiAgICAgICAgbGV0IHRva2VucyA9IHZlYyFbMnUzMiwgMTAsIDQyLCA3LCAxNSwgMywgOF07CiAgICAgICAgbGV0IHN0YXJ0ICA9IHN0ZDo6dGltZTo6SW5zdGFudDo6bm93KCk7CiAgICAgICAgbGV0IGl0ZXJzICA9IDEwMDA7CiAgICAgICAgZm9yIF8gaW4gMC4uaXRlcnMgewogICAgICAgICAgICBsZXQgXyA9IG1vZGVsLnByZWRpY3RfbmV4dCgmdG9rZW5zKTsKICAgICAgICB9CiAgICAgICAgbGV0IGVsYXBzZWQgPSBzdGFydC5lbGFwc2VkKCk7CiAgICAgICAgcHJpbnRsbiEoIkluZmVyZW5jZTogezouM31tcy9xdWVyeSAoe30gcXVlcmllcykiLAogICAgICAgICAgICAgICAgIGVsYXBzZWQuYXNfc2Vjc19mNjQoKSAqIDEwMDAuMCAvIGl0ZXJzIGFzIGY2NCwgaXRlcnMpOwogICAgfQp9Cg==').decode(),
}
import zipfile, io
buf = io.BytesIO()
with zipfile.ZipFile(buf,'w') as zf:
    for n,s in RUST_FILES.items(): zf.writestr(f'phase_snn_rs/{n}',s)
open(os.path.join(PLATFORM['ckpt_dir'],'phase_snn_rs.zip'),'wb').write(buf.getvalue())
print('Rust source exported')
